# CoDA-GAT preprocessing (cleaned)

Part 1 is the original acquisition, label and expression-loading code, kept verbatim except for removed dead cells (see `NOTEBOOK_CLEANUP.md`). Part 2 calls the `coda_preproc` package, whose output was checked against the archived August 2026 handoff: all 26 node features in all 8 folds match (max |diff| 6e-8).

**Provenance warning.** Part 1 depends on live services (GEO, BioStudies, MyGene.info). The supplied notebook run (8 Oct 2026) produced 10,732 common genes and a different GSE240729 matrix from the archived package (10,735 genes), so a fresh run is not guaranteed to reproduce the archived matrices. Archive `interim/expr_cache/` and the `reports/expr_cache_manifest.json` written below.

## Part 1: acquisition, labels, per-cohort expression

In [ ]:
# ============================================================
# CELL 1 — Environment  [v2]
# Colab (Python 3.13) already ships numpy, pandas, scipy, sklearn,
# networkx, pyarrow, openpyxl, tqdm. Do NOT reinstall or pin them —
# the old pins had no 3.13 wheels and forced slow source builds.
# Only three packages are genuinely missing.
#
# >>> DO NOT CANCEL. inmoose may take 2-3 minutes to build. <
# ============================================================
import sys
print(f"Python {sys.version.split()[0]}\n")

!pip install GEOparse
!pip install mygene
!pip install inmoose

import importlib
NEED = {"GEOparse": "GEOparse", "mygene": "mygene", "inmoose": "inmoose",
        "networkx": "networkx", "pyarrow": "pyarrow", "openpyxl": "openpyxl",
        "tqdm": "tqdm", "sklearn": "scikit-learn", "scipy": "scipy",
        "numpy": "numpy", "pandas": "pandas"}

print("\n" + "="*50)
missing = []
for mod, pkg in NEED.items():
    try:
        m = importlib.import_module(mod)
        print(f"  OK   {mod:12s} {getattr(m, '__version__', '?')}")
    except Exception as e:
        print(f"  FAIL {mod:12s} {type(e).__name__}")
        missing.append(pkg)

if missing:
    raise RuntimeError(
        f"MISSING: {missing}\n"
        f"Run: !pip install {' '.join(missing)}\n"
        f"Then restart the runtime (Runtime > Restart session) and re-run "
        f"Cell 1 before anything else. Do NOT cancel the install.")
print("\nAll imports OK — proceed to Cell 2.")

In [ ]:
# ============================================================
# CELL 2 — Imports, Drive, Paths, Config
# ============================================================
import os, re, json, gzip, time, hashlib, warnings, tarfile, shutil
from pathlib import Path
from collections import namedtuple, defaultdict, Counter

import numpy as np
import pandas as pd
import networkx as nx
from tqdm.auto import tqdm

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 200)

from google.colab import drive
drive.mount("/content/drive", force_remount=False)

# ---- Single source of truth for every path ------------------

ROOT = Path("/content/drive/MyDrive/CoDAGAT_preprocessing_fix_v1")

PATHS = {
    "root":      ROOT,
    "raw":       ROOT / "raw",           # untouched downloads
    "raw_geo":   ROOT / "raw" / "geo",
    "raw_gtex":  ROOT / "raw" / "gtex",
    "raw_string":ROOT / "raw" / "string",
    "raw_nano":  ROOT / "raw" / "nanostring",
    "interim":   ROOT / "interim",       # stage outputs
    "processed": ROOT / "processed",     # final artifacts
    "reports":   ROOT / "reports",       # audit + integrity reports
    "config":    ROOT / "config",
    "manifest":  ROOT / "manifests",
}
for p in PATHS.values():
    p.mkdir(parents=True, exist_ok=True)

# ---- Global constants (locked by spec) ----------------------
RANDOM_SEED       = 42
SEEDS             = [42, 43, 44, 45, 46]
STRING_THRESHOLD  = 700
N_PANEL_GENES     = 72
GTEX_MIN_SD       = 0.1

np.random.seed(RANDOM_SEED)

# ---- Manifest helper: makes every stage idempotent ----------
def sha256_file(path, blocksize=1 << 20):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(blocksize), b""):
            h.update(chunk)
    return h.hexdigest()

def sha256_obj(obj):
    return hashlib.sha256(
        json.dumps(obj, sort_keys=True, default=str).encode()
    ).hexdigest()

def write_manifest(stage, inputs, params, outputs):
    """Record what produced what, so re-runs can be skipped and
    so every number in the paper is traceable to a config."""
    man = {
        "stage": stage,
        "timestamp": time.strftime("%Y-%m-%dT%H:%M:%S"),
        "inputs": inputs,
        "params": params,
        "outputs": {
            str(o): (sha256_file(o) if Path(o).exists() else None)
            for o in outputs
        },
    }
    fp = PATHS["manifest"] / f"manifest_{stage}.json"
    fp.write_text(json.dumps(man, indent=2))
    print(f"[manifest] wrote {fp.name}")
    return man

def stage_is_current(stage, outputs):
    """True if all outputs exist and manifest hashes still match."""
    fp = PATHS["manifest"] / f"manifest_{stage}.json"
    if not fp.exists():
        return False
    man = json.loads(fp.read_text())
    for o in outputs:
        o = Path(o)
        if not o.exists():
            return False
        if man["outputs"].get(str(o)) != sha256_file(o):
            return False
    return True

print(f"\nROOT = {ROOT}")
print(f"Free space: {shutil.disk_usage('/content').free / 1e9:.1f} GB on /content")
for k, v in PATHS.items():
    v.mkdir(parents=True, exist_ok=True)
    assert v.exists(), f"could not create {k}: {v} — is Drive mounted?"
print("all PATHS verified:", {k: str(v) for k, v in PATHS.items()})


In [ ]:
# ============================================================
# CELL 3 — COHORT REGISTRY  (v2, post-tracker audit)
#   T1 = LODO primary   (train / calibrate / TEST)
#   T2 = training-only  (train only)
#   T3 = sensitivity    (train only, reported separately)
#   T4 = external       (never enters pipeline)
#   AUDIT / EXCLUDED / REJECTED = logged, not usable
# ============================================================

COHORTS = {
    # ---------------- CONFIRMED T1 ----------------
    "GSE135251": dict(tier="T1", platform="rnaseq", etiology="NAFLD", repo="GEO",
                      preservation="frozen", n_expected=206, status="confirmed"),
    "GSE84044":  dict(tier="T1", platform="microarray", etiology="HBV", gpl="GPL570", repo="GEO",
                      preservation="frozen", n_expected=124, status="confirmed"),
    "GSE162694": dict(tier="T1", platform="rnaseq", etiology="NAFLD", repo="GEO",
                      preservation="frozen", n_expected=112, status="confirmed"),
    "GSE130970": dict(tier="T1", platform="rnaseq", etiology="NAFLD", repo="GEO",
                      preservation="frozen", n_expected=78, status="confirmed"),
    "E-MTAB-6863": dict(tier="T1", platform="rnaseq", etiology="HCV_NASH_mixed",
                      repo="ArrayExpress", preservation="frozen", n_expected=69,
                      status="new_accepted",
                      note="F0=12 F1=15 F2=15 F3=14 F4=13. Exact staging. "
                           "Best F2 density available; adds HCV etiology."),
    "GSE89632":  dict(tier="T1", platform="microarray", etiology="NAFLD", gpl="GPL14951",
                      repo="GEO", preservation="frozen", n_expected=55, status="confirmed",
                      note="marginal: 10 positives under F2+"),
    "GSE48452":  dict(tier="T1", platform="microarray", etiology="NAFLD", gpl="GPL11532",
                      repo="GEO", preservation="frozen", n_expected=60, status="confirmed",
                      note="NOT viable as T1 for F2+ (4 positives); T1 for early-vs-adv only"),
    "GSE31803":  dict(tier="T1", platform="microarray", etiology="NAFLD", gpl="GPL570",
                      repo="GEO", preservation="frozen", n_expected=72, status="confirmed",
                      note="BOUNDARY-DEGENERATE: grouped F0-1 vs F3-4, no F1/F2 present"),

    # ---------------- T2 training-only ----------------
    "GSE276114": dict(tier="T2", platform="rnaseq", etiology="MASLD_mixed", repo="GEO",
                      preservation="frozen", n_expected=177, status="confirmed",
                      note="39 samples grouped_F0_F2 -> ambiguous across F2 boundary"),
    "GSE240729": dict(tier="T2", platform="rnaseq", etiology="MASLD", repo="GEO",
                      preservation="FFPE", n_expected=67, status="confirmed"),
    "E-MTAB-9815": dict(tier="T2", platform="rnaseq", etiology="NAFLD",
                      repo="ArrayExpress", preservation="frozen", n_expected=58,
                      status="new_accepted",
                      note="Groups: NAFL / NASH F0-2 / NASH F3-4. F0-2 bin straddles "
                           "the F2 boundary -> grouped_F0_F2, training only."),

    # ---------------- T3 sensitivity ----------------
    "GSE213621": dict(tier="T3", platform="rnaseq", etiology="MASLD", repo="GEO",
                      preservation="frozen", n_expected=368, status="RELABEL_REQUIRED",
                      note="CRITICAL: Phase 2 recovered only 107 F2. Published work using "
                           "this series reports n=368 with F0-F2 vs F3-F4 labels. "
                           "Staging exists under a metadata key the parser missed. "
                           "Recover before tiering: may promote to T1."),

    # ---------------- T4 external ----------------
    "GSE163211": dict(tier="T4", platform="nanostring", etiology="NAFLD_spectrum", repo="GEO",
                      preservation="frozen", n_expected=318, status="confirmed",
                      note="EXTERNAL ONLY. Coarse labels. Cannot validate F2+ boundary."),

    # ---------------- AUDIT ----------------
    "GSE193066": dict(tier="AUDIT", platform="rnaseq", etiology="NAFLD", repo="GEO",
                      n_expected=164, status="candidate",
                      note="106 baseline + 58 follow-up biopsies. LONGITUDINAL DUPLICATES: "
                           "keep one biopsy per patient. Staging unverified."),
    "GSE159088": dict(tier="AUDIT", platform="microarray", etiology="NAFLD", repo="GEO",
                      n_expected=80, status="candidate",
                      note="Fibrosis staging not visible in public sample fields. Low priority."),
    "GSE207310": dict(tier="AUDIT", platform="rnaseq", etiology="NAFLD", repo="GEO",
                      n_expected=30, status="candidate",
                      note="Kleiner grades in metadata; Phase 2 TAR extraction empty. "
                           "Retry per-GSM table fallback."),

    # ---------------- REJECTED (logged, never used) ----------------
    "GSE49541":  dict(tier="REJECTED", platform="microarray", gpl="GPL570", repo="GEO",
                      n_expected=72, status="rejected",
                      reason="SUSPECTED DUPLICATE of GSE31803: both 72 samples, GPL570, "
                             "NAFLD, 40 mild(F0-1) / 32 advanced(F3-4). Adding it would "
                             "place identical patients in train and test. "
                             "Verify by GSM overlap before any reconsideration."),
    "GSE167523": dict(tier="REJECTED", platform="rnaseq", etiology="NAFLD", repo="GEO",
                      n_expected=98, status="rejected",
                      reason="No F0-F4 staging. 48 mild NAFLD / 50 NASH = disease category, "
                             "not fibrosis stage. Stage must never be inferred from NAFL/NASH."),
    "GSE126848": dict(tier="REJECTED", repo="GEO", status="rejected",
                      reason="No fibrosis staging"),
    "GSE61260":  dict(tier="REJECTED", repo="GEO", status="rejected",
                      reason="0 staged samples recovered in Phase 2"),
    "GSE151158": dict(tier="REJECTED", platform="nanostring", repo="GEO", status="rejected",
                      reason="NASH activity panel, not F0-F4 staging"),
    "GSE139602": dict(tier="REJECTED", platform="rnaseq", repo="GEO", status="rejected",
                      reason="Cirrhosis/ACLF focused; no early-stage distribution"),
    "GSE202379": dict(tier="REJECTED", repo="GEO", status="rejected",
                      reason="single-nucleus RNA-seq; bulk-only criterion"),
    "GSE270584": dict(tier="REJECTED", repo="GEO", status="rejected",
                      reason="spatial/single-cell, non-human signal"),
    "GSE115193": dict(tier="REJECTED", repo="GEO", n_expected=3, status="rejected",
                      reason="only 3 samples"),
    "E-MTAB-12445": dict(tier="REJECTED", repo="ArrayExpress", status="rejected",
                      reason="NAFLD vs healthy; no fibrosis staging"),
    "GSE134146": dict(tier="REJECTED", repo="GEO", status="rejected",
                      reason="circRNA; not bulk staged transcriptomics"),
    "GSE94799":  dict(tier="REJECTED", repo="GEO", status="rejected",
                      reason="no confirmed fibrosis-stage labels"),
}

TIER_RULES = {          # can_train, can_calibrate, can_test
    "T1": (True, True, True),   "T2": (True, False, False),
    "T3": (True, False, False), "T4": (False, False, False),
    "AUDIT": (False, False, False),
    "REJECTED": (False, False, False),
}

REPO_OF = {g: m.get("repo", "GEO") for g, m in COHORTS.items()}

def tier_of(gse):            return COHORTS[gse]["tier"]
def cohorts_by_tier(*tiers): return [g for g, m in COHORTS.items() if m["tier"] in tiers]

(PATHS["config"] / "cohorts.json").write_text(json.dumps(COHORTS, indent=2))

print("REGISTRY v2")
for t in ["T1", "T2", "T3", "T4", "AUDIT", "REJECTED"]:
    ms = cohorts_by_tier(t)
    print(f"  {t:9s} n={len(ms):2d}  {ms}")

usable = [g for g in COHORTS if tier_of(g) in ("T1", "T2", "T3")]
print(f"\nProjected N_real (T1+T2+T3): "
      f"{sum(COHORTS[g].get('n_expected', 0) for g in usable)}")
print("  (assumes GSE213621 relabel recovers 368; if it stays 107, subtract 261)")
print(f"\nRepositories to handle: {sorted(set(REPO_OF[g] for g in usable))}")
print("!! Cell 5 needs an ArrayExpress fetcher for E-MTAB-6863 / E-MTAB-9815")

In [ ]:
# ============================================================
# CELL 4 — GUARDS
# The three non-negotiable rules, as assertions. No bypass flag.
# ============================================================

class GuardViolation(AssertionError):
    pass

SYN_PREFIX = "SYN_"

def assert_no_synthetic(df, split_name, fold_id="NA"):
    """RULE 2: calibration and test must be raw."""
    idx = pd.Index(df.index if df.index.name in (None, "sample_id") else df["sample_id"])
    bad = [s for s in idx.astype(str) if s.startswith(SYN_PREFIX)]
    if bad:
        raise GuardViolation(
            f"FOLD {fold_id}: {len(bad)} synthetic samples found in '{split_name}'. "
            f"Examples: {bad[:5]}. Augmentation must run AFTER the "
            f"proper-train/calibration split, on proper-train only."
        )
    return True

def assert_cohort_disjoint(train_df, cal_df, test_df, fold_id="NA"):
    """RULE 1: test cohort appears nowhere else."""
    tr, ca, te = set(train_df.cohort), set(cal_df.cohort), set(test_df.cohort)
    if len(te) != 1:
        raise GuardViolation(f"FOLD {fold_id}: test must be exactly ONE cohort, got {te}")
    if te & tr:
        raise GuardViolation(f"FOLD {fold_id}: test cohort {te & tr} also in proper-train")
    if te & ca:
        raise GuardViolation(f"FOLD {fold_id}: test cohort {te & ca} also in calibration")
    return True

def assert_no_subject_overlap(train_df, test_df, fold_id="NA"):
    """Silent leakage: GEO super-series and re-analyses share patients."""
    if "subject_id" in train_df.columns and "subject_id" in test_df.columns:
        a = set(train_df.subject_id.dropna())
        b = set(test_df.subject_id.dropna())
        if a & b:
            raise GuardViolation(
                f"FOLD {fold_id}: {len(a & b)} subjects appear in both train and test: "
                f"{list(a & b)[:5]}")
    return True

def assert_synthetic_provenance(syn_df):
    """RULE 3: synthesis is within-cohort and within-class."""
    req = {"source_cohort_a", "source_cohort_b", "source_stage_a", "source_stage_b",
           "cohort", "stage", "method"}
    missing = req - set(syn_df.columns)
    if missing:
        raise GuardViolation(f"synthetic frame missing provenance columns: {missing}")
    bad_c = syn_df[(syn_df.cohort != syn_df.source_cohort_a) |
                   (syn_df.cohort != syn_df.source_cohort_b)]
    if len(bad_c):
        raise GuardViolation(f"{len(bad_c)} synthetic samples cross cohort boundaries")
    bad_s = syn_df[(syn_df.stage != syn_df.source_stage_a) |
                   (syn_df.stage != syn_df.source_stage_b)]
    if len(bad_s):
        raise GuardViolation(f"{len(bad_s)} synthetic samples cross class boundaries")
    return True

def assert_tier_usage(df, split_name, fold_id="NA"):
    """T2/T3 never calibrate or test. T4/AUDIT/EXCLUDED never enter at all."""
    slot = {"proper_train": 0, "calibration": 1, "test": 2}[split_name]
    for coh in set(df.cohort):
        allowed = TIER_RULES[tier_of(coh)][slot]
        if not allowed:
            raise GuardViolation(
                f"FOLD {fold_id}: cohort {coh} (tier {tier_of(coh)}) "
                f"is not permitted in split '{split_name}'")
    return True

def assert_no_healthy_as_F0(labels_df):
    """Healthy liver is not stage F0. Conflating them leaks the deviation signal."""
    if "resolution" not in labels_df.columns:
        raise GuardViolation("labels frame has no 'resolution' column")
    bad = labels_df[(labels_df.resolution == "healthy") & (labels_df.stage.notna())]
    if len(bad):
        raise GuardViolation(f"{len(bad)} healthy samples were assigned a fibrosis stage")
    return True

def assert_tensor_integrity(arr, name="tensor"):
    if np.isnan(arr).any():
        raise GuardViolation(f"{name}: {int(np.isnan(arr).sum())} NaN values")
    if np.isinf(arr).any():
        raise GuardViolation(f"{name}: {int(np.isinf(arr).sum())} Inf values")
    return True


# ---------------- self-test ----------------
def _test_guards():
    ok = 0
    tr = pd.DataFrame({"cohort": ["GSE135251"]*3}, index=["a","b","c"])
    ca = pd.DataFrame({"cohort": ["GSE84044"]*2},  index=["d","e"])
    te = pd.DataFrame({"cohort": ["GSE130970"]*2}, index=["f","g"])
    assert assert_cohort_disjoint(tr, ca, te); ok += 1

    bad_cal = pd.DataFrame({"cohort": ["GSE84044"]*2}, index=["SYN_1","e"])
    try:
        assert_no_synthetic(bad_cal, "calibration"); print("  FAIL: synthetic not caught")
    except GuardViolation:
        ok += 1

    leaky = pd.DataFrame({"cohort": ["GSE130970"]*2}, index=["x","y"])
    try:
        assert_cohort_disjoint(leaky, ca, te); print("  FAIL: overlap not caught")
    except GuardViolation:
        ok += 1

    try:
        assert_tier_usage(pd.DataFrame({"cohort": ["GSE240729"]}), "test")
        print("  FAIL: T2-as-test not caught")
    except GuardViolation:
        ok += 1

    try:
        assert_tensor_integrity(np.array([1.0, np.nan])); print("  FAIL: NaN not caught")
    except GuardViolation:
        ok += 1

    print(f"guard self-test: {ok}/5 passed")

_test_guards()
def check_expression(df, name):
    if not isinstance(df, pd.DataFrame) or df.empty:
        raise ValueError(f"{name}: missing or empty matrix")

    if not df.index.is_unique or not df.columns.is_unique:
        raise ValueError(f"{name}: duplicate gene/sample IDs")

    if df.index.hasnans or df.columns.hasnans:
        raise ValueError(f"{name}: missing gene/sample IDs")

    if not np.isfinite(df.to_numpy(dtype=float)).all():
        raise ValueError(f"{name}: NaN or infinite expression values")

    return df

In [ ]:
# ============================================================
# CELL 5 — DOWNLOAD
# ============================================================
try:
    import GEOparse
except ModuleNotFoundError:
    raise RuntimeError(
        "GEOparse missing. Run Cell 1 and let it FINISH (do not cancel). "
        "Then restart the runtime and re-run Cells 1, 2, 3, 4, then 5.")

import urllib.request, urllib.error
import GEOparse
import urllib.request, urllib.error

GEO_FTP = "https://ftp.ncbi.nlm.nih.gov/geo/series/{stub}nnn/{gse}/"

def _stub(gse):
    n = gse.replace("GSE", "")
    return "GSE" + (n[:-3] if len(n) > 3 else "")

def download_file(url, dest, retries=3, timeout=180):
    dest = Path(dest)
    if dest.exists() and dest.stat().st_size > 0:
        return dest, "cached"
    dest.parent.mkdir(parents=True, exist_ok=True)
    for attempt in range(retries):
        try:
            urllib.request.urlretrieve(url, dest)
            if dest.stat().st_size == 0:
                raise IOError("zero-byte download")
            return dest, "downloaded"
        except Exception as e:
            if dest.exists():
                dest.unlink()
            if attempt == retries - 1:
                return None, f"failed: {type(e).__name__}: {e}"
            time.sleep(3 * (attempt + 1))
    return None, "failed"

def verify_gzip(path):
    """A truncated gzip that half-reads is a failure mode you already hit."""
    try:
        with gzip.open(path, "rb") as f:
            while f.read(1 << 22):
                pass
        return True
    except Exception:
        return False

def fetch_cohort(gse, pull_supp=True):
    outdir = PATHS["raw_geo"] / gse
    outdir.mkdir(parents=True, exist_ok=True)
    rec = {"gse": gse, "tier": tier_of(gse), "soft": None,
           "supp": [], "errors": []}

    # --- SOFT (metadata; always needed, even for EXCLUDED cohorts) ---
    try:
        gseobj = GEOparse.get_GEO(geo=gse, destdir=str(outdir), silent=True,
                                  include_data=False)
        rec["soft"] = "ok"
        rec["n_gsm"] = len(gseobj.gsms)
        rec["gpl"] = list(gseobj.gpls.keys())
    except Exception as e:
        rec["errors"].append(f"SOFT: {type(e).__name__}: {e}")
        return rec

    # --- supplementary files ---
    if pull_supp:
        base = GEO_FTP.format(stub=_stub(gse), gse=gse) + "suppl/"
        try:
            listing = urllib.request.urlopen(base, timeout=60).read().decode("utf-8", "ignore")
            fnames = sorted(set(re.findall(r'href="([^"]+)"', listing)))
            fnames = [f for f in fnames if not f.startswith(("/", "?"))]
        except Exception as e:
            rec["errors"].append(f"suppl listing: {e}")
            fnames = []

        for fn in fnames:
            # skip giant RAW tarballs unless the cohort needs them
            if fn.endswith("_RAW.tar") and gse not in ("GSE207310", "GSE163211"):
                rec["supp"].append({"file": fn, "status": "skipped_RAW"})
                continue
            dest, status = download_file(base + fn, outdir / fn)
            entry = {"file": fn, "status": status}
            if dest and fn.endswith(".gz"):
                entry["gzip_ok"] = verify_gzip(dest)
                if not entry["gzip_ok"]:
                    dest.unlink(missing_ok=True)
                    entry["status"] = "corrupt_removed"
            if dest and dest.exists():
                entry["bytes"] = dest.stat().st_size
            rec["supp"].append(entry)
    return rec


# ---------------- run ----------------
DOWNLOAD_LOG = {}
targets = [g for g, m in COHORTS.items()]      # EVERY cohort, no exclusions

for gse in tqdm(targets, desc="GEO"):
    DOWNLOAD_LOG[gse] = fetch_cohort(gse)

(PATHS["reports"] / "download_log.json").write_text(json.dumps(DOWNLOAD_LOG, indent=2))

print("\nDOWNLOAD SUMMARY")
print(f"{'cohort':12s} {'tier':9s} {'GSM':>5s}  {'supp':>4s}  errors")
for g, r in DOWNLOAD_LOG.items():
    n_ok = sum(1 for s in r.get("supp", []) if s.get("status") in ("downloaded", "cached"))
    print(f"{g:12s} {r['tier']:9s} {r.get('n_gsm', 0):5d}  {n_ok:4d}  "
          f"{'; '.join(r['errors'])[:60]}")

write_manifest("p01_download",
               inputs={"registry": sha256_obj(COHORTS)},
               params={"pull_supp": True},
               outputs=[PATHS["reports"] / "download_log.json"])

In [ ]:
# ============================================================
# CELL 6 — ARRAYEXPRESS / BIOSTUDIES FETCHER
# For E-MTAB-6863 (T1) and E-MTAB-9815 (T2).
# Different API from GEO: metadata lives in the SDRF file,
# expression in a processed-data file whose name varies per study.
# NOTE: endpoint layout changes occasionally. This tries several
# and prints exactly what it found — read the output before trusting it.
# ============================================================

BIOSTUDIES_API  = "https://www.ebi.ac.uk/biostudies/api/v1/studies/{acc}"
BIOSTUDIES_FILE = "https://www.ebi.ac.uk/biostudies/files/{acc}/{fname}"
AE_FILES_ALT    = "https://www.ebi.ac.uk/biostudies/api/v1/files/{acc}/{fname}"

def _http_json(url, timeout=90):
    req = urllib.request.Request(url, headers={"Accept": "application/json"})
    with urllib.request.urlopen(req, timeout=timeout) as r:
        return json.loads(r.read().decode("utf-8", "ignore"))

def _walk_files(node, out):
    """BioStudies nests file entries in section/subsections; flatten them."""
    if isinstance(node, dict):
        if "path" in node and "size" in node:
            out.append(node)
        for v in node.values():
            _walk_files(v, out)
    elif isinstance(node, list):
        for v in node:
            _walk_files(v, out)
    return out

def fetch_arrayexpress(acc):
    outdir = PATHS["raw_geo"] / acc
    outdir.mkdir(parents=True, exist_ok=True)
    rec = {"acc": acc, "tier": tier_of(acc), "files": [], "errors": []}

    # ---- 1. study metadata + file listing ----
    try:
        meta = _http_json(BIOSTUDIES_API.format(acc=acc))
        (outdir / f"{acc}_biostudies.json").write_text(json.dumps(meta, indent=2))
    except Exception as e:
        rec["errors"].append(f"api: {type(e).__name__}: {e}")
        return rec

    entries = _walk_files(meta, [])
    names = sorted({e["path"] for e in entries})
    rec["listing"] = names
    print(f"\n[{acc}] {len(names)} files listed")
    for n in names[:40]:
        print("   ", n)
    if len(names) > 40:
        print(f"    ... and {len(names)-40} more")

    # ---- 2. always take SDRF + IDF (metadata) ----
    want = [n for n in names if n.lower().endswith((".sdrf.txt", ".idf.txt"))]

    # ---- 3. processed expression: name varies, so pattern-match ----
    expr_pat = re.compile(
        r"(count|cpm|fpkm|tpm|expression|processed|matrix|norm)", re.I)
    want += [n for n in names
             if expr_pat.search(n)
             and n.lower().endswith((".txt", ".tsv", ".csv", ".gz", ".zip"))
             and not n.lower().endswith((".sdrf.txt", ".idf.txt"))]
    want = sorted(set(want))

    for fname in want:
        got = None
        for tmpl in (BIOSTUDIES_FILE, AE_FILES_ALT):
            url = tmpl.format(acc=acc, fname=urllib.parse.quote(fname))
            dest, status = download_file(url, outdir / Path(fname).name)
            if dest:
                got = {"file": fname, "status": status,
                       "bytes": dest.stat().st_size, "url": url}
                if fname.endswith(".gz"):
                    got["gzip_ok"] = verify_gzip(dest)
                break
        rec["files"].append(got or {"file": fname, "status": "failed_all_endpoints"})

    return rec


import urllib.parse
AE_COHORTS = [g for g, m in COHORTS.items()
              if m.get("repo") == "ArrayExpress" and m["tier"] in ("T1", "T2", "T3")]

for acc in AE_COHORTS:
    DOWNLOAD_LOG[acc] = fetch_arrayexpress(acc)

(PATHS["reports"] / "download_log.json").write_text(json.dumps(DOWNLOAD_LOG, indent=2))

print("\nARRAYEXPRESS SUMMARY")
for acc in AE_COHORTS:
    r = DOWNLOAD_LOG[acc]
    ok = [f for f in r["files"] if f and f.get("status") in ("downloaded", "cached")]
    print(f"  {acc:14s} files_ok={len(ok):2d}  errors={r['errors']}")
    for f in ok:
        print(f"      {f['file']:55s} {f['bytes']/1e6:8.2f} MB")

print("\nACTION: confirm an SDRF file downloaded for each accession.")
print("        The SDRF is where per-sample fibrosis stage lives.")

In [ ]:
# ============================================================
# CELL 7 — STAGE PARSER  (the Phase 2 bug lives here)
#
# Returns (stage, resolution, raw) — NEVER a bare stage.
# The bare regex \b([0-4])\b as a fallback is what fabricated
# exact stages for 111 grouped-label samples. It is not used.
# ============================================================

Label = namedtuple("Label", ["stage", "resolution", "raw"])

RES_EXACT       = "exact"
RES_G_F0F1      = "grouped_F0_F1"
RES_G_F3F4      = "grouped_F3_F4"
RES_G_F0F2      = "grouped_F0_F2"
RES_HEALTHY     = "healthy"
RES_UNKNOWN     = "unknown"

STAGE_KEYS = [
    "fibrosis stage", "fibrosis_stage", "fibrosis-stage", "fibrotic stage",
    "fibrosis score", "fibrosis_score", "fibrosis", "stage",
    "nas fibrosis", "nas_fibrosis", "nafld fibrosis stage",
    "scheuer score", "scheuer", "metavir", "metavir score",
    "kleiner fibrosis_grade", "kleiner fibrosis grade", "kleiner",
    "ishak fibrosis", "ishak", "knodell fibrosis", "brunt",
    "disease group", "group", "diagnosis", "histology", "condition",
    "fibrosis severity", "severity", "characteristics",
]

HEALTHY_WORDS = ("healthy", "normal liver", "normal control", "control",
                 "donor", "non-diseased", "nondiseased")

# --- ordered patterns; FIRST MATCH WINS -----------------------
GROUPED_PATTERNS = [
    # explicit ranges, with or without the letter F
    (re.compile(r"f?\s*0\s*[-–—to]+\s*f?\s*1\b", re.I), "F0", RES_G_F0F1),
    (re.compile(r"f?\s*3\s*[-–—to]+\s*f?\s*4\b", re.I), "F3", RES_G_F3F4),
    (re.compile(r"f?\s*0\s*[-–—to]+\s*f?\s*2\b", re.I), "F0", RES_G_F0F2),
    (re.compile(r"f?\s*1\s*[-–—to]+\s*f?\s*4\b", re.I), None, RES_UNKNOWN),   # straddles
    # word forms only when a bracketed range accompanies them
    (re.compile(r"mild.*?stage\s*0\s*[-–—]\s*1", re.I),     "F0", RES_G_F0F1),
    (re.compile(r"advanced.*?stage\s*3\s*[-–—]\s*4", re.I), "F3", RES_G_F3F4),
    (re.compile(r"mild.*?\bf?0\s*[-–—]\s*f?1\b", re.I),     "F0", RES_G_F0F1),
    (re.compile(r"advanced.*?\bf?3\s*[-–—]\s*f?4\b", re.I), "F3", RES_G_F3F4),
]

RE_F_EXACT     = re.compile(r"\bf\s*([0-4])\b", re.I)
RE_STAGE_EXACT = re.compile(r"(?:stage|score|grade)\s*[:=]?\s*([0-4])(?![\d.])", re.I)
RE_KLEINER_SUB = re.compile(r"\b([0-4])\s*([abc])\b", re.I)
RE_ISHAK       = re.compile(r"ishak[^0-9]{0,12}([0-6])", re.I)

ISHAK_MAP = {0: "F0", 1: "F1", 2: "F1", 3: "F2", 4: "F3", 5: "F4", 6: "F4"}


def parse_stage(raw_value, key_hint=""):
    """raw_value: the characteristics string. key_hint: its metadata key."""
    if raw_value is None:
        return Label(None, RES_UNKNOWN, "")
    raw = str(raw_value).strip()
    if not raw:
        return Label(None, RES_UNKNOWN, raw)
    low = f"{key_hint} {raw}".lower()

    # 1. healthy FIRST — healthy liver is never stage F0
    if any(w in low for w in HEALTHY_WORDS):
        if not re.search(r"\b(f[0-4]|stage\s*[0-4])\b", low):
            return Label(None, RES_HEALTHY, raw)

    # 2. grouped ranges BEFORE any single-digit extraction
    for pat, stage, res in GROUPED_PATTERNS:
        if pat.search(low):
            return Label(stage, res, raw)

    # 3. Ishak (different scale, must be caught before generic digits)
    m = RE_ISHAK.search(low)
    if m:
        return Label(ISHAK_MAP[int(m.group(1))], "exact_ishak_mapped", raw)

    # 4. Kleiner sub-stages 1A/1B/1C -> F1
    m = RE_KLEINER_SUB.search(raw)
    if m and ("kleiner" in low or "grade" in low or "fibrosis" in low):
        return Label(f"F{int(m.group(1))}", RES_EXACT, raw)

    # 5. explicit F0..F4
    m = RE_F_EXACT.search(raw)
    if m:
        return Label(f"F{int(m.group(1))}", RES_EXACT, raw)

    # 6. "stage: 2" / "score = 3" — requires the keyword, never a bare digit
    m = RE_STAGE_EXACT.search(low)
    if m:
        return Label(f"F{int(m.group(1))}", RES_EXACT, raw)

    # 7. a bare digit ONLY if the metadata key itself is a fibrosis key
    if any(k in key_hint.lower() for k in
           ("fibrosis", "fibrotic", "metavir", "scheuer", "kleiner", "ishak")):
        m = re.fullmatch(r"\s*([0-4])(?:\.0)?\s*", raw)
        if m:
            return Label(f"F{int(m.group(1))}", RES_EXACT, raw)

    # 8. NEVER guess. NAFL/NASH/steatosis are diagnoses, not stages.
    return Label(None, RES_UNKNOWN, raw)


USABLE = {
    # resolution -> (ok_for_F2plus_task, ok_for_ordinal, ok_for_early_vs_adv)
    RES_EXACT:            (True,  True,  True),
    "exact_ishak_mapped": (True,  True,  True),
    RES_G_F0F1:           (True,  False, True),
    RES_G_F3F4:           (True,  False, True),
    RES_G_F0F2:           (False, False, True),   # straddles the F2 boundary
    RES_HEALTHY:          (False, False, False),
    RES_UNKNOWN:          (False, False, False),
}


# ---------------- self-test on REAL strings from your Phase 2 run ----------------
def _test_parser():
    cases = [
        # (key, value, expected_stage, expected_resolution)
        ("Stage",                   "mild (fibrosis stage 0-1)",     "F0", RES_G_F0F1),
        ("Stage",                   "advanced (fibrosis stage 3-4)", "F3", RES_G_F3F4),
        ("disease group",           "F0-2",                          "F0", RES_G_F0F2),
        ("disease group",           "F3",                            "F3", RES_EXACT),
        ("disease group",           "F4",                            "F4", RES_EXACT),
        ("fibrotic stage",          "2",                             "F2", RES_EXACT),
        ("fibrosis stage",          "3",                             "F3", RES_EXACT),
        ("nas_fibrosis",            "0",                             "F0", RES_EXACT),
        ("kleiner fibrosis_grade",  "1A",                            "F1", RES_EXACT),
        ("kleiner fibrosis_grade",  "1B",                            "F1", RES_EXACT),
        ("Scheuer score",           "S2",                            None, RES_UNKNOWN),
        ("fibrosis stage",          "F2",                            "F2", RES_EXACT),
        ("diagnosis",               "NAFL",                          None, RES_UNKNOWN),
        ("diagnosis",               "NASH",                          None, RES_UNKNOWN),
        ("tissue",                  "healthy liver",                 None, RES_HEALTHY),
        ("condition",               "normal control",                None, RES_HEALTHY),
        ("group",                   "steatosis",                     None, RES_UNKNOWN),
        ("ishak fibrosis",          "ishak 5",                       "F4", "exact_ishak_mapped"),
        ("fibrosis",                "1-4",                           None, RES_UNKNOWN),
        ("age",                     "52",                            None, RES_UNKNOWN),  # bare digit
    ]
    fails = []
    for key, val, exp_s, exp_r in cases:
        got = parse_stage(val, key)
        if (got.stage, got.resolution) != (exp_s, exp_r):
            fails.append((key, val, exp_s, exp_r, got.stage, got.resolution))
    print(f"parser self-test: {len(cases)-len(fails)}/{len(cases)} passed")
    for f in fails:
        print(f"  FAIL key={f[0]!r} val={f[1]!r}  expected=({f[2]},{f[3]})  got=({f[4]},{f[5]})")
    return not fails

_test_parser()

In [ ]:
# ============================================================
# CELL 8 — METADATA KEY DISCOVERY
# Dumps every characteristics key + value distribution per cohort.
# Purpose: find staging keys the parser doesn't know about yet.
# GSE213621 is the priority — published work reports n=368 with
# F0-F2 / F3-F4 labels; Phase 2 recovered only 107 F2.
# ============================================================

def harvest_characteristics(gse):
    """Return {key: Counter(values)} across all GSMs in a series."""
    softdir = PATHS["raw_geo"] / gse
    try:
        g = GEOparse.get_GEO(geo=gse, destdir=str(softdir), silent=True, include_data=False)
    except Exception as e:
        return None, f"{type(e).__name__}: {e}"

    keys = defaultdict(Counter)
    per_sample = {}
    for gsm_id, gsm in g.gsms.items():
        row = {}
        for field, vals in gsm.metadata.items():
            if not field.startswith(("characteristics", "title", "source_name",
                                     "description", "supplementary")):
                continue
            for v in vals:
                if ":" in v:
                    k, val = v.split(":", 1)
                    k, val = k.strip().lower(), val.strip()
                else:
                    k, val = field, v.strip()
                keys[k][val] += 1
                row[k] = val
        per_sample[gsm_id] = row
    return (keys, per_sample), None


DISCOVERY = {}
targets = cohorts_by_tier("T1", "T2", "T3", "T4", "AUDIT")
targets = [t for t in targets if REPO_OF[t] == "GEO"]

for gse in tqdm(targets, desc="metadata"):
    res, err = harvest_characteristics(gse)
    DISCOVERY[gse] = {"error": err} if err else {
        "keys": {k: dict(c.most_common(25)) for k, c in res[0].items()},
        "n_gsm": len(res[1]),
        "per_sample": res[1],
    }

# --- print, prioritising the cohort we need to recover ---
def show(gse, max_keys=40):
    d = DISCOVERY.get(gse, {})
    if d.get("error"):
        print(f"{gse}: ERROR {d['error']}"); return
    print(f"\n{'='*72}\n{gse}  (n_gsm={d['n_gsm']})")
    for k, vals in list(d["keys"].items())[:max_keys]:
        total = sum(vals.values())
        looks_staging = any(s in k for s in
            ("fibro", "stage", "metavir", "scheuer", "kleiner", "ishak",
             "grade", "group", "nas", "histo", "severity", "diagnos"))
        flag = " <<< POSSIBLE STAGING" if looks_staging else ""
        preview = ", ".join(f"{v}({n})" for v, n in list(vals.items())[:8])
        print(f"  [{total:4d}] {k:34s} {preview[:110]}{flag}")

show("GSE213621", max_keys=60)          # priority
for gse in targets:
    if gse != "GSE213621":
        show(gse)

with open(PATHS["reports"] / "metadata_keys.json", "w") as f:
    json.dump({g: {k: v for k, v in d.items() if k != "per_sample"}
               for g, d in DISCOVERY.items()}, f, indent=2)

print("\n" + "="*72)
print("ACTION: scan the <<< POSSIBLE STAGING lines above.")
print("Any key the parser does not already handle goes into STAGE_KEYS (Cell 7),")
print("and any new value format goes into the parser test cases BEFORE re-running.")

In [ ]:
# ============================================================
# CELL 9 — BUILD LABEL TABLE
# One row per sample. Resolution class is mandatory.
# Nothing is dropped — unusable samples are retained and flagged.
# ============================================================

def build_labels_for_cohort(gse):
    d = DISCOVERY.get(gse)
    if not d or d.get("error"):
        return pd.DataFrame()
    meta = COHORTS[gse]
    rows = []
    for gsm_id, row in d["per_sample"].items():
        best = Label(None, RES_UNKNOWN, "")
        # try dedicated staging keys first, then everything else
        ordered = ([k for k in row if any(s in k for s in
                    ("fibro", "metavir", "scheuer", "kleiner", "ishak"))]
                   + [k for k in row if k not in
                      [j for j in row if any(s in j for s in
                       ("fibro", "metavir", "scheuer", "kleiner", "ishak"))]])
        for k in ordered:
            lab = parse_stage(row[k], k)
            if lab.resolution != RES_UNKNOWN:
                best = lab
                if lab.resolution in (RES_EXACT, "exact_ishak_mapped"):
                    break        # exact beats grouped; stop looking
        rows.append(dict(
            sample_id=gsm_id, cohort=gse,
            stage=best.stage, resolution=best.resolution, raw_string=best.raw,
            etiology=meta.get("etiology"), platform=meta.get("platform"),
            preservation=meta.get("preservation"), tier=meta["tier"],
            repo=meta.get("repo", "GEO"), subject_id=None,
        ))
    return pd.DataFrame(rows)


frames = [build_labels_for_cohort(g) for g in targets]
labels_raw = pd.concat([f for f in frames if len(f)], ignore_index=True)

# ---- task labels, derived strictly from resolution ----
def f2plus(r):
    if not USABLE[r.resolution][0] or r.stage is None: return np.nan
    return 1 if r.stage in ("F2", "F3", "F4") else 0

def early_adv(r):
    if not USABLE[r.resolution][2] or r.stage is None: return np.nan
    return 1 if r.stage in ("F3", "F4") else 0

def ordinal(r):
    if not USABLE[r.resolution][1] or r.stage is None: return np.nan
    return int(r.stage[1])

labels_raw["y_T1_significant"] = labels_raw.apply(f2plus, axis=1)
labels_raw["y_T3_early_adv"]   = labels_raw.apply(early_adv, axis=1)
labels_raw["y_T2_ordinal"]     = labels_raw.apply(ordinal, axis=1)

assert_no_healthy_as_F0(labels_raw)

out = PATHS["interim"] / "labels_raw.parquet"
labels_raw.to_parquet(out, index=False)

# ---------------- reports ----------------
print("RESOLUTION BREAKDOWN")
print(pd.crosstab(labels_raw.cohort, labels_raw.resolution, margins=True))

print("\nSTAGE x COHORT (exact + grouped)")
print(pd.crosstab(labels_raw.cohort, labels_raw.stage.fillna("NONE"), margins=True))

print("\nTASK USABILITY per cohort")
summ = labels_raw.groupby("cohort").agg(
    n=("sample_id", "size"),
    T1_usable=("y_T1_significant", lambda s: s.notna().sum()),
    T1_pos=("y_T1_significant", lambda s: (s == 1).sum()),
    T2_usable=("y_T2_ordinal", lambda s: s.notna().sum()),
    T3_usable=("y_T3_early_adv", lambda s: s.notna().sum()),
    T3_pos=("y_T3_early_adv", lambda s: (s == 1).sum()),
)
summ["T1_minority"] = summ[["T1_pos"]].join(
    (summ.T1_usable - summ.T1_pos).rename("T1_neg")).min(axis=1)
summ["T1_tier_suggestion"] = np.where(summ.T1_minority >= 15, "T1-eligible", "T2/T3 only")
print(summ.to_string())

print(f"\nN_real (any usable task): "
      f"{labels_raw[['y_T1_significant','y_T3_early_adv']].notna().any(axis=1).sum()}")
print(f"Unknown/unparsed: {(labels_raw.resolution == RES_UNKNOWN).sum()}")
print(f"Healthy (excluded from staging): {(labels_raw.resolution == RES_HEALTHY).sum()}")

write_manifest("p03_labels",
               inputs={"registry": sha256_obj(COHORTS)},
               params={"parser": "v2_resolution_classes"},
               outputs=[out])

In [ ]:
# ============================================================
# CELL 10 — PARSER v3
# Fixes found in the Cell 8/9 audit:
#   (a) concatenated grouped forms: F0F1, F3F4, F0F2
#   (b) "Control"/"HC"/"Normal" as a VALUE -> healthy
#   (c) Scheuer S (fibrosis) must beat Scheuer G (inflammation)
#   (d) "nas score" / "activity score" must NOT satisfy the
#       generic (stage|score|grade)\s*[0-4] pattern
#   (e) cohort-level healthy override from a separate field
# ============================================================

# --- keys that are FIBROSIS, in priority order (first wins) ---
FIBROSIS_KEY_PRIORITY = [
    "scheuer score s",          # Scheuer S = fibrosis stage  (NOT "scheuer score g")
    "fibrosis stage", "fibrosis_stage", "fibrosis-stage",
    "fibrotic stage", "fibrosisscore", "fibrosis score",
    "fibrosis (stage)", "fibrosis",
    "kleiner fibrosis_grade", "kleiner fibrosis grade",
    "metavir", "metavir score", "ishak fibrosis", "ishak",
    "disease group", "stage",
]

# --- keys that look like staging but are NOT fibrosis ---
NON_FIBROSIS_KEYS = {
    "scheuer score g",                    # necroinflammatory grade
    "nas score", "nas", "nafld activity score", "nafld activity_score",
    "lobular inflammation grade", "lobular inflammation (severity)",
    "cytological ballooning grade", "ballooning (intensity)",
    "steatosis grade", "steatosis (%)", "inflammation", "fat",
    "aspartate transaminase (u/l)", "alanine transaminase (u/l)",
    "alkaline phosphatase (u/l)", "age", "age (y)", "age at biopsy",
    "bmi", "body mass index (kg/m2)", "batch", "washbatch",
    "steatosis, activity,_and_fibrosis_score",   # SAF composite, not F-stage
}

# --- values that mean "not a patient with staged fibrosis" ---
HEALTHY_VALUES = {
    "control", "controls", "hc", "healthy control", "normal",
    "normal liver", "normal liver histology", "healthy", "donor",
    "non-diseased", "nondiseased",
}
HEALTHY_FIELDS = ["disease", "diagnosis", "group", "source_name_ch1",
                  "fibrotic stage", "condition", "tissue"]

# --- concatenated + separated grouped ranges (checked BEFORE digits) ---
GROUPED_PATTERNS = [
    (re.compile(r"\bf?0\s*[-–—/to]*\s*f?1\b", re.I), "F0", RES_G_F0F1),
    (re.compile(r"\bf?3\s*[-–—/to]*\s*f?4\b", re.I), "F3", RES_G_F3F4),
    (re.compile(r"\bf?0\s*[-–—/to]*\s*f?2\b", re.I), "F0", RES_G_F0F2),
    (re.compile(r"\bf?1\s*[-–—/to]*\s*f?4\b", re.I), None, RES_UNKNOWN),  # straddles
    (re.compile(r"mild.*?stage\s*0\s*[-–—]\s*1", re.I),     "F0", RES_G_F0F1),
    (re.compile(r"advanced.*?stage\s*3\s*[-–—]\s*4", re.I), "F3", RES_G_F3F4),
]

RE_SCHEUER_GS = re.compile(r"\bg([0-4])\s*s([0-4])\b", re.I)   # "G1S2" -> S is fibrosis


def parse_stage(raw_value, key_hint=""):
    raw = "" if raw_value is None else str(raw_value).strip()
    key = str(key_hint).strip().lower()
    val = raw.lower()
    unknown = Label(None, RES_UNKNOWN, raw)

    if not raw or key in NON_FIBROSIS_KEYS:
        return unknown

    if val in HEALTHY_VALUES or any(
        re.fullmatch(rf"{re.escape(h)}s?", val)
        for h in HEALTHY_VALUES
    ):
        return Label(None, RES_HEALTHY, raw)

    if any(h in val for h in (
        "normal liver", "healthy control", "non-diseased"
    )):
        if re.search(r"\bf\s*\d|\bstage\s*\d", val):
            return unknown
        return Label(None, RES_HEALTHY, raw)

    # Handle the declared Ishak scale before generic numeric stages.
    if "ishak" in key or "ishak" in val:
        m = re.fullmatch(
            r"(?:ishak(?:\s+(?:fibrosis|score|stage))*"
            r"\s*[:=]?\s*)?([0-6])(?:\.0)?",
            val,
        )
        if m:
            return Label(
                ISHAK_MAP[int(m[1])], "exact_ishak_mapped", raw
            )
        return unknown

    m = re.fullmatch(r"g([0-4])\s*s([0-4])", val)
    if m:
        return Label(f"F{m[2]}", RES_EXACT, raw)

    v = val
    m = re.fullmatch(r"(?:mild|advanced)\s*\((.*)\)", v)
    if m:
        v = m[1].strip()

    v = re.sub(
        r"^(?:fibrosis\s+(?:stage|score)|fibrotic\s+stage|stage)"
        r"\s*[:=]?\s*",
        "",
        v,
    )

    # Match the complete range, never its first stage.
    m = re.fullmatch(
        r"f?\s*([0-4])\s*(?:[-–—/_]|to)\s*f?\s*([0-4])",
        v,
    )
    if m is None:
        m = re.fullmatch(r"f\s*([0-4])\s*f\s*([0-4])", v)

    if m:
        grouped = {
            (0, 1): ("F0", RES_G_F0F1),
            (0, 2): ("F0", RES_G_F0F2),
            (3, 4): ("F3", RES_G_F3F4),
        }
        stage, resolution = grouped.get(
            (int(m[1]), int(m[2])),
            (None, RES_UNKNOWN),
        )
        return Label(stage, resolution, raw)

    if "kleiner" in key or "fibro" in key:
        if re.fullmatch(r"1\s*[abc]", v):
            return Label("F1", RES_EXACT, raw)

    m = re.fullmatch(r"f\s*([0-4])(?:\.0)?", v)
    if m:
        return Label(f"F{m[1]}", RES_EXACT, raw)

    if (
        key in FIBROSIS_KEY_PRIORITY
        or "fibro" in key
        or key.startswith("scheuer score s")
    ):
        m = re.fullmatch(r"([0-4])(?:\.0)?", v)
        if m:
            return Label(f"F{m[1]}", RES_EXACT, raw)

    return unknown
for value, key, expected_stage, expected_resolution in [
    ("F1-F2", "fibrosis stage", None, RES_UNKNOWN),
    ("F2.5", "fibrosis stage", None, RES_UNKNOWN),
    ("F0-F1-F2", "fibrosis stage", None, RES_UNKNOWN),
    ("F0 to F2", "fibrosis stage", "F0", RES_G_F0F2),
    ("3", "ishak fibrosis", "F2", "exact_ishak_mapped"),
    ("34", "fibrosis stage", None, RES_UNKNOWN),
    ("F0 F1", "fibrosis stage", "F0", RES_G_F0F1),
]:
    got = parse_stage(value, key)
    assert (got.stage, got.resolution) == (
        expected_stage, expected_resolution
    ), (value, key, got)
def _test_parser_v3():
    cases = [
        ("fibrotic stage",         "F2",     "F2", RES_EXACT),
        ("fibrotic stage",         "F0F1",   "F0", RES_G_F0F1),   # was BROKEN
        ("fibrotic stage",         "F3F4",   "F3", RES_G_F3F4),   # was BROKEN
        ("fibrotic stage",         "Control", None, RES_HEALTHY), # was BROKEN
        ("scheuer score s",        "2",      "F2", RES_EXACT),
        ("scheuer score g",        "2",      None, RES_UNKNOWN),  # was BROKEN
        ("nas score",              "0",      None, RES_UNKNOWN),  # was BROKEN
        ("nafld activity score",   "5",      None, RES_UNKNOWN),
        ("fibrosis stage",         "normal liver histology", None, RES_HEALTHY),  # was BROKEN
        ("fibrosis stage",         "0",      "F0", RES_EXACT),
        ("disease group",          "F0-2",   "F0", RES_G_F0F2),
        ("disease group",          "F4",     "F4", RES_EXACT),
        ("stage",                  "mild (fibrosis stage 0-1)",     "F0", RES_G_F0F1),
        ("stage",                  "advanced (fibrosis stage 3-4)", "F3", RES_G_F3F4),
        ("stage",                  "mild",   None, RES_UNKNOWN),   # bare word, no range
        ("kleiner fibrosis_grade", "1A",     "F1", RES_EXACT),
        ("kleiner fibrosis_grade", "2",      "F2", RES_EXACT),
        ("fibrosisscore",          "F2",     "F2", RES_EXACT),
        ("fibrosis",               "3.5",    None, RES_UNKNOWN),
        ("diagnosis",              "HC",     None, RES_HEALTHY),
        ("diagnosis",              "NASH",   None, RES_UNKNOWN),
        ("group",                  "Healthy obese", None, RES_UNKNOWN),
        ("age",                    "52",     None, RES_UNKNOWN),
        ("nafld stage",            "NASH_F1_F4", None, RES_UNKNOWN),
    ]
    fails = [(k, v, es, er, g.stage, g.resolution)
             for k, v, es, er in cases
             for g in [parse_stage(v, k)]
             if (g.stage, g.resolution) != (es, er)]
    print(f"parser v3 self-test: {len(cases)-len(fails)}/{len(cases)} passed")
    for f in fails:
        print(f"  FAIL key={f[0]!r} val={f[1]!r} expected=({f[2]},{f[3]}) got=({f[4]},{f[5]})")
    return not fails

assert _test_parser_v3(), "Parser tests failed — DO NOT PROCEED."

In [ ]:
# ============================================================
# CELL 11 — REBUILD LABEL TABLE  (supersedes Cell 9)
# ============================================================

# Treat "Healthy obese" (GSE48452) as diseased-spectrum, not healthy.
# Flip to True only if you decide otherwise — and say so in the paper.
EXCLUDE_HEALTHY_OBESE = False

# Fields that override any fibrosis value with "this is a control subject"
COHORT_HEALTHY_OVERRIDE = {
    "GSE135251": ("disease",        {"control"}),
    "GSE89632":  ("diagnosis",      {"hc"}),
    "GSE48452":  ("group",          {"control"} | ({"healthy obese"} if EXCLUDE_HEALTHY_OBESE else set())),
    "GSE213621": ("fibrotic stage", {"control"}),
    "GSE162694": ("fibrosis stage", {"normal liver histology"}),
}

# Patient-level dedup: keep one biopsy per subject
DEDUP_RULES = {
    "GSE193066": ("biopsy", "1st biopsy"),   # drop 58 follow-up biopsies
}


def pick_stage(row_dict, gse):
    """Try fibrosis keys in declared priority order; exact beats grouped."""
    best = Label(None, RES_UNKNOWN, "")
    keys_present = list(row_dict.keys())
    ordered = ([k for k in FIBROSIS_KEY_PRIORITY if k in row_dict]
               + [k for k in keys_present
                  if k not in FIBROSIS_KEY_PRIORITY and k not in NON_FIBROSIS_KEYS])
    for k in ordered:
        lab = parse_stage(row_dict[k], k)
        if lab.resolution in (RES_EXACT, "exact_ishak_mapped"):
            return lab
        if lab.resolution != RES_UNKNOWN and best.resolution == RES_UNKNOWN:
            best = lab
    return best


def build_labels_v3(gse):
    d = DISCOVERY.get(gse)
    if not d or d.get("error"):
        return pd.DataFrame()
    meta = COHORTS[gse]
    ov_field, ov_values = COHORT_HEALTHY_OVERRIDE.get(gse, (None, set()))
    dd_field, dd_keep = DEDUP_RULES.get(gse, (None, None))

    rows = []
    for gsm_id, row in d["per_sample"].items():
        # --- dedup: drop repeat biopsies of the same patient ---
        if dd_field and str(row.get(dd_field, "")).strip().lower() != dd_keep.lower():
            rows.append(dict(sample_id=gsm_id, cohort=gse, stage=None,
                             resolution="duplicate_biopsy", raw_string=row.get(dd_field, ""),
                             etiology=meta.get("etiology"), platform=meta.get("platform"),
                             preservation=meta.get("preservation"), tier=meta["tier"],
                             repo=meta.get("repo", "GEO"), subject_id=gsm_id))
            continue

        # --- healthy override beats any fibrosis value ---
        if ov_field and str(row.get(ov_field, "")).strip().lower() in ov_values:
            lab = Label(None, RES_HEALTHY, str(row.get(ov_field, "")))
        else:
            lab = pick_stage(row, gse)

        rows.append(dict(
            sample_id=gsm_id, cohort=gse,
            stage=lab.stage, resolution=lab.resolution, raw_string=lab.raw,
            etiology=meta.get("etiology"), platform=meta.get("platform"),
            preservation=meta.get("preservation"), tier=meta["tier"],
            repo=meta.get("repo", "GEO"), subject_id=None,
        ))
    return pd.DataFrame(rows)


USABLE["duplicate_biopsy"] = (False, False, False)

labels_raw = pd.concat([build_labels_v3(g) for g in targets], ignore_index=True)
labels_raw = labels_raw[labels_raw.cohort.map(lambda c: tier_of(c) != "REJECTED")]

labels_raw["y_T1_significant"] = labels_raw.apply(f2plus, axis=1)
labels_raw["y_T3_early_adv"]   = labels_raw.apply(early_adv, axis=1)
labels_raw["y_T2_ordinal"]     = labels_raw.apply(ordinal, axis=1)

assert_no_healthy_as_F0(labels_raw)
labels_raw.to_parquet(PATHS["interim"] / "labels_raw.parquet", index=False)

# ---------------- reports ----------------
print("RESOLUTION BREAKDOWN (v3)")
print(pd.crosstab(labels_raw.cohort, labels_raw.resolution, margins=True))

print("\nSTAGE x COHORT")
print(pd.crosstab(labels_raw.cohort, labels_raw.stage.fillna("NONE"), margins=True))

summ = labels_raw.groupby("cohort").agg(
    n_gsm=("sample_id", "size"),
    healthy=("resolution", lambda s: (s == RES_HEALTHY).sum()),
    dup=("resolution", lambda s: (s == "duplicate_biopsy").sum()),
    unk=("resolution", lambda s: (s == RES_UNKNOWN).sum()),
    T1_usable=("y_T1_significant", lambda s: s.notna().sum()),
    T1_pos=("y_T1_significant", lambda s: (s == 1).sum()),
    T2_usable=("y_T2_ordinal", lambda s: s.notna().sum()),
    T3_usable=("y_T3_early_adv", lambda s: s.notna().sum()),
    T3_pos=("y_T3_early_adv", lambda s: (s == 1).sum()),
)
summ["T1_neg"] = summ.T1_usable - summ.T1_pos
summ["T1_minority"] = summ[["T1_pos", "T1_neg"]].min(axis=1)
summ["suggest"] = np.where(summ.T1_minority >= 15, "T1-eligible",
                   np.where(summ.T1_usable >= 20, "T2 (train-only)", "review"))
print("\nTASK USABILITY (v3)")
print(summ.to_string())

print("\nRETIER ACTIONS (registry says X, data says Y):")
for c, r in summ.iterrows():
    cur = tier_of(c)
    want = "T1" if r.suggest == "T1-eligible" else ("T2" if "T2" in r.suggest else "review")
    if cur in ("T1", "T2", "T3", "AUDIT") and cur != want and want != "review":
        print(f"  {c:12s} registry={cur:6s} -> data suggests {want}"
              f"  (usable={int(r.T1_usable)}, minority={int(r.T1_minority)})")

n_real = labels_raw[["y_T1_significant", "y_T3_early_adv"]].notna().any(axis=1).sum()
print(f"\nN_real (usable in at least one task): {n_real}")
print(f"  healthy excluded: {(labels_raw.resolution == RES_HEALTHY).sum()}")
print(f"  duplicate biopsies dropped: {(labels_raw.resolution == 'duplicate_biopsy').sum()}")

write_manifest("p03_labels", inputs={"registry": sha256_obj(COHORTS)},
               params={"parser": "v3", "exclude_healthy_obese": EXCLUDE_HEALTHY_OBESE},
               outputs=[PATHS["interim"] / "labels_raw.parquet"])

In [ ]:
# ============================================================
# CELL 12 — PARSER PATCH v3.1
# BUG: separator was optional, so bare "34" matched F3-F4 and
#      "01" matched F0-F1. Now require F-prefix OR a real separator.
# ALSO: T4 (NanoString) must never go through this parser.
# ============================================================

GROUPED_PATTERNS = [
    # F-prefixed: F0F1, F0-F1, F0_F1, f0 f1
    (re.compile(r"\bf\s*0\s*[-–—/_ ]*\s*f?\s*1\b", re.I), "F0", RES_G_F0F1),
    (re.compile(r"\bf\s*3\s*[-–—/_ ]*\s*f?\s*4\b", re.I), "F3", RES_G_F3F4),
    (re.compile(r"\bf\s*0\s*[-–—/_ ]*\s*f?\s*2\b", re.I), "F0", RES_G_F0F2),
    (re.compile(r"\bf\s*1\s*[-–—/_ ]*\s*f?\s*4\b", re.I), None, RES_UNKNOWN),
    # bare digits: EXPLICIT separator required
    (re.compile(r"\b0\s*[-–—/]\s*1\b"), "F0", RES_G_F0F1),
    (re.compile(r"\b3\s*[-–—/]\s*4\b"), "F3", RES_G_F3F4),
    (re.compile(r"\b0\s*[-–—/]\s*2\b"), "F0", RES_G_F0F2),
    (re.compile(r"\b1\s*[-–—/]\s*4\b"), None, RES_UNKNOWN),
    (re.compile(r"mild.*?stage\s*0\s*[-–—]\s*1", re.I),     "F0", RES_G_F0F1),
    (re.compile(r"advanced.*?stage\s*3\s*[-–—]\s*4", re.I), "F3", RES_G_F3F4),
]

def _test_patch():
    cases = [("34", None), ("age 34", None), ("01", None), ("1.34", None),
             ("52", None), ("F0F1", "F0"), ("F3F4", "F3"), ("F0-2", "F0"),
             ("f0-f1", "F0"), ("0-1", "F0"), ("3-4", "F3"),
             ("mild (fibrosis stage 0-1)", "F0"),
             ("advanced (fibrosis stage 3-4)", "F3"), ("NASH_F1_F4", None)]
    bad = []
    for v, exp in cases:
        got = next((s for p, s, r in GROUPED_PATTERNS if p.search(v.lower())), None)
        if got != exp:
            bad.append((v, exp, got))
    print(f"patch test: {len(cases)-len(bad)}/{len(cases)} passed")
    for b in bad:
        print("  FAIL", b)
    return not bad

assert _test_patch(), "Patch failed — STOP."
assert _test_parser_v3(), "Full parser suite failed — STOP."

# ---- T4 never enters the staging parser ----
PARSE_TARGETS = [g for g in targets if tier_of(g) != "T4"]
print(f"parsing {len(PARSE_TARGETS)} cohorts (T4 excluded: "
      f"{[g for g in targets if tier_of(g)=='T4']})")

labels_raw = pd.concat([build_labels_v3(g) for g in PARSE_TARGETS], ignore_index=True)
labels_raw = labels_raw[labels_raw.cohort.map(lambda c: tier_of(c) != "REJECTED")]
labels_raw["y_T1_significant"] = labels_raw.apply(f2plus, axis=1)
labels_raw["y_T3_early_adv"]   = labels_raw.apply(early_adv, axis=1)
labels_raw["y_T2_ordinal"]     = labels_raw.apply(ordinal, axis=1)
assert_no_healthy_as_F0(labels_raw)
labels_raw.to_parquet(PATHS["interim"] / "labels_raw.parquet", index=False)

print("\nSTAGE x COHORT (v3.1)")
print(pd.crosstab(labels_raw.cohort, labels_raw.stage.fillna("NONE"), margins=True))

summ = labels_raw.groupby("cohort").agg(
    n_gsm=("sample_id","size"),
    healthy=("resolution", lambda s:(s==RES_HEALTHY).sum()),
    dup=("resolution", lambda s:(s=="duplicate_biopsy").sum()),
    unk=("resolution", lambda s:(s==RES_UNKNOWN).sum()),
    T1_usable=("y_T1_significant", lambda s:s.notna().sum()),
    T1_pos=("y_T1_significant", lambda s:(s==1).sum()),
    T2_usable=("y_T2_ordinal", lambda s:s.notna().sum()),
    T3_usable=("y_T3_early_adv", lambda s:s.notna().sum()),
    T3_pos=("y_T3_early_adv", lambda s:(s==1).sum()))
summ["T1_neg"] = summ.T1_usable - summ.T1_pos
summ["T1_minority"] = summ[["T1_pos","T1_neg"]].min(axis=1)
print("\nTASK USABILITY (v3.1)")
print(summ.to_string())

print("\nSANITY vs published Table I:")
for c, exp in [("GSE84044","43/20/33/18/10"), ("GSE135251","38/47/53/54/14"),
               ("GSE162694","35/30/27/8/12"), ("GSE130970","25/28/9/14/2")]:
    d = labels_raw[labels_raw.cohort==c].stage.value_counts()
    got = "/".join(str(int(d.get(f"F{i}",0))) for i in range(5))
    print(f"  {c:12s} expect {exp:16s} got {got:16s} {'OK' if got==exp else '<<< MISMATCH'}")

In [ ]:
# ============================================================
# CELL 13 — TIER PROMOTION + F2 AUDIT + LABEL-SHIFT TABLE
# Tier is now set by DATA, not by the hand-written guess.
# Rule: minority class >= 15 under F2+ -> T1. Else T2.
# ============================================================

MIN_MINORITY_T1 = 15

PROMOTIONS = {
    "GSE213621": ("T3", "T1", "192 samples recovered (F0F1/F3F4); 97 neg / 202 pos"),
    "GSE193066": ("AUDIT","T1", "106 baseline biopsies; 43 neg / 63 pos. NOTE: PLS high-risk enriched"),
    "GSE240729": ("T2", "T1", "26 neg / 41 pos — binary-evaluable"),
    "GSE207310": ("AUDIT","T2", "only 8 positives"),
    "GSE48452":  ("T1", "T2", "3 positives under F2+ — not binary-evaluable"),
    "GSE89632":  ("T1", "T2", "10 positives under F2+ — below threshold"),
    "GSE159088": ("AUDIT","REJECTED", "no fibrosis staging in metadata (80/80 unknown)"),
}

for gse,(old,new,why) in PROMOTIONS.items():
    assert COHORTS[gse]["tier"] == old, f"{gse}: expected {old}, found {COHORTS[gse]['tier']}"
    COHORTS[gse]["tier"] = new
    COHORTS[gse]["retier_reason"] = why
    print(f"  {gse:12s} {old:6s} -> {new:8s}  {why}")

labels_raw["tier"] = labels_raw.cohort.map(tier_of)
labels_raw = labels_raw[labels_raw.tier != "REJECTED"]
labels_raw.to_parquet(PATHS["interim"] / "labels_raw.parquet", index=False)
(PATHS["config"] / "cohorts.json").write_text(json.dumps(COHORTS, indent=2))

# ---------------- F2+ audit ----------------
aud = (labels_raw.dropna(subset=["y_T1_significant"])
       .groupby(["tier","cohort"])["y_T1_significant"]
       .agg(n="size", pos="sum").reset_index())
aud["neg"] = aud.n - aud.pos
aud["pos_pct"] = 100*aud.pos/aud.n
aud = aud.sort_values(["tier","n"], ascending=[True,False])
print("\nF2+ AUDIT\n", aud.to_string(index=False))

# ---------------- per-fold label shift (JS) ----------------
def js_div(p,q):
    p,q = np.asarray(p,float), np.asarray(q,float); m=(p+q)/2
    kl = lambda a,b: np.sum([x*np.log2(x/y) for x,y in zip(a,b) if x>0])
    return 0.5*kl(p,m)+0.5*kl(q,m)

pool = labels_raw.dropna(subset=["y_T1_significant"])
shift = []
for f in aud.loc[aud.tier=="T1","cohort"]:
    te = pool[pool.cohort==f].y_T1_significant
    tr = pool[pool.cohort!=f].y_T1_significant
    mt = [1-te.mean(), te.mean()]; mr = [1-tr.mean(), tr.mean()]
    shift.append(dict(fold=f, n=len(te), test_pos=mt[1], train_pos=mr[1],
                      JS=js_div(mt,mr)))
shift = pd.DataFrame(shift).sort_values("JS", ascending=False)
print("\nLABEL SHIFT PER FOLD (independent variable of the dose-response experiment)")
print(shift.to_string(index=False, float_format=lambda x: f"{x:.4f}"))
shift.to_csv(PATHS["reports"]/"label_shift_per_fold.csv", index=False)

# ---------------- cohort-label confounding ----------------
from sklearn.metrics import normalized_mutual_info_score as nmi
print(f"\nNMI(cohort, F2+ label) = "
      f"{nmi(pool.cohort, pool.y_T1_significant.astype(int)):.3f}")
print(f"JS range = {shift.JS.min():.4f} to {shift.JS.max():.4f} "
      f"({shift.JS.max()/shift.JS.min():.0f}x)")

print(f"\nT1 folds: {len(shift)}   T1 samples: {int(shift.n.sum())}")
print(f"N_real (usable in >=1 task): "
      f"{labels_raw[['y_T1_significant','y_T3_early_adv']].notna().any(axis=1).sum()}")

write_manifest("p03b_tiers", inputs={"labels": "labels_raw.parquet"},
               params={"min_minority_T1": MIN_MINORITY_T1, "promotions": PROMOTIONS},
               outputs=[PATHS["interim"]/"labels_raw.parquet",
                        PATHS["reports"]/"label_shift_per_fold.csv"])

In [ ]:
# ============================================================
# CELL 14 — ARRAYEXPRESS SDRF -> LABELS
# SDRF is a TSV, one row per assay. Fibrosis stage sits in a
# "Characteristics[...]" or "Factor Value[...]" column whose exact
# name varies per study. We discover, then parse with the SAME
# parser used for GEO — no separate code path, no separate bugs.
# ============================================================

def read_sdrf(acc):
    d = PATHS["raw_geo"] / acc
    cands = sorted(d.glob("*.sdrf.txt")) + sorted(d.glob("*sdrf*"))
    if not cands:
        print(f"  {acc}: NO SDRF FOUND in {d}")
        return None
    df = pd.read_csv(cands[0], sep="\t", dtype=str)
    print(f"  {acc}: {cands[0].name}  shape={df.shape}")
    return df

def sdrf_key(col):
    """'Characteristics[fibrosis stage]' -> 'fibrosis stage'"""
    m = re.search(r"\[(.+?)\]", col)
    return (m.group(1) if m else col).strip().lower()

AE_LABEL_FRAMES = {}
for acc in [g for g, m in COHORTS.items()
            if m.get("repo") == "ArrayExpress" and m["tier"] in ("T1","T2","T3")]:
    sdrf = read_sdrf(acc)
    if sdrf is None:
        continue

    # ---- show every candidate column so you can eyeball it ----
    cand = [c for c in sdrf.columns
            if c.lower().startswith(("characteristics", "factor value"))]
    print(f"\n  [{acc}] candidate annotation columns:")
    for c in cand:
        vals = sdrf[c].dropna().value_counts()
        flag = " <<< POSSIBLE STAGING" if any(
            s in sdrf_key(c) for s in ("fibro","stage","metavir","ishak",
                                        "kleiner","scheuer","grade","disease")) else ""
        print(f"      {sdrf_key(c):32s} {dict(list(vals.items())[:6])}{flag}")

    # ---- sample identifier column ----
    id_col = next((c for c in ["Source Name","Assay Name","Comment[ENA_RUN]",
                               "Scan Name","Extract Name"] if c in sdrf.columns), None)
    if id_col is None:
        print(f"  {acc}: no usable ID column, columns={list(sdrf.columns)[:12]}")
        continue

    rows = []
    meta = COHORTS[acc]
    for _, r in sdrf.iterrows():
        row_dict = {sdrf_key(c): r[c] for c in cand if pd.notna(r[c])}
        lab = pick_stage(row_dict, acc)
        rows.append(dict(
            sample_id=str(r[id_col]).strip(), cohort=acc,
            stage=lab.stage, resolution=lab.resolution, raw_string=lab.raw,
            etiology=meta.get("etiology"), platform=meta.get("platform"),
            preservation=meta.get("preservation"), tier=meta["tier"],
            repo="ArrayExpress", subject_id=None))
    f = pd.DataFrame(rows).drop_duplicates(subset="sample_id")
    AE_LABEL_FRAMES[acc] = f
    print(f"  {acc}: {len(f)} unique assays  ->  "
          f"{f.resolution.value_counts().to_dict()}")
    print(f"      stages: {f.stage.value_counts().to_dict()}")

# ---- merge into labels_raw ----
if AE_LABEL_FRAMES:
    ae = pd.concat(AE_LABEL_FRAMES.values(), ignore_index=True)
    ae["y_T1_significant"] = ae.apply(f2plus, axis=1)
    ae["y_T3_early_adv"]   = ae.apply(early_adv, axis=1)
    ae["y_T2_ordinal"]     = ae.apply(ordinal, axis=1)

    labels_raw = pd.concat(
        [labels_raw[~labels_raw.cohort.isin(AE_LABEL_FRAMES)], ae],
        ignore_index=True)
    assert_no_healthy_as_F0(labels_raw)
    labels_raw.to_parquet(PATHS["interim"] / "labels_raw.parquet", index=False)

    print("\nSTAGE x COHORT (with ArrayExpress)")
    print(pd.crosstab(labels_raw.cohort, labels_raw.stage.fillna("NONE"), margins=True))
    print(f"\nN_real: "
          f"{labels_raw[['y_T1_significant','y_T3_early_adv']].notna().any(axis=1).sum()}")
    print("\nEXPECT E-MTAB-6863 ~ F0=12 F1=15 F2=15 F3=14 F4=13")
else:
    print("\nNo ArrayExpress SDRF parsed — check Cell 6 downloaded the .sdrf.txt files.")

In [ ]:
# ============================================================
# CELL 14b — LOADERS, GPL MAPPING, AND SAMPLE-ID MATCHING
# Run after Cells 2 and 5, before Cell 15.
# _clean_symbol() is defined in Cell 15 before _gpl_map() is used.
# ============================================================

import GEOparse
print("GEOparse imported successfully")
import re
from pathlib import Path
from collections import defaultdict
from urllib.parse import urlsplit

import GEOparse
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

_GPL_CACHE = globals().get("_GPL_CACHE", {})

# Create the cache directory without deleting existing files.
cache = PATHS["interim"] / "expr_cache"
cache.mkdir(parents=True, exist_ok=True)


# ---------- GPL annotation tables ----------

def _gpl_tables(gse):
    """Return all non-empty GPL tables for a series."""
    key = f"{gse}::all"

    if key in _GPL_CACHE:
        return _GPL_CACHE[key]

    out = []

    try:
        g = GEOparse.get_GEO(
            geo=gse,
            destdir=str(PATHS["raw_geo"] / gse),
            silent=True,
            include_data=False,
        )

        for gpl_id, gpl in g.gpls.items():
            table = getattr(gpl, "table", None)

            if (
                table is not None
                and len(table.columns) > 0
                and len(table) > 0
            ):
                out.append((gpl_id, table))

    except Exception as exc:
        print(f"      GPL load failed {gse}: {exc}")
        return []

    # Do not cache failed/empty retrievals.
    if out:
        _GPL_CACHE[key] = out

    return out


def _gpl_map(ids, gse):
    tables = _gpl_tables(gse)

    if not tables:
        return {}

    ids_s = set(map(str, ids))
    prefer = [
        "gene symbol", "gene_symbol", "symbol", "genesymbol",
        "ilmn_gene", "gene", "gene_assignment", "gene assignment",
    ]

    best_map = {}
    best_cov = 0.0
    best_desc = None

    for gpl_id, table in tables:
        id_col = "ID" if "ID" in table.columns else table.columns[0]
        cols = {str(c).strip().lower(): c for c in table.columns}

        for candidate in prefer:
            if candidate not in cols:
                continue

            mapping = {
                str(k): _clean_symbol(v)
                for k, v in zip(table[id_col], table[cols[candidate]])
            }
            mapping = {k: v for k, v in mapping.items() if v}

            coverage = len(ids_s & set(mapping)) / max(len(ids_s), 1)

            if coverage > best_cov:
                best_map = mapping
                best_cov = coverage
                best_desc = f"{gpl_id}:{cols[candidate]}"

    if best_desc:
        print(
            f"      GPL {best_desc} -> {len(best_map)} probes, "
            f"coverage {best_cov:.1%}"
        )

    return best_map


# ---------- Sample-ID matching ----------

def _tokens(value):
    return {
        token
        for token in re.split(r"[^A-Za-z0-9]+", str(value))
        if len(token) >= 4
    }


def build_gsm_token_index(gse):
    """Retain exact metadata aliases only when they identify one GSM."""
    idx = defaultdict(set)
    exact_candidates = defaultdict(set)

    try:
        g = GEOparse.get_GEO(
            geo=gse,
            destdir=str(PATHS["raw_geo"] / gse),
            silent=True,
            include_data=False,
        )
    except Exception as exc:
        print(f"      Sample metadata failed {gse}: {exc}")
        return idx, {}

    for gsm_id, gsm in g.gsms.items():
        for key in (
            "title", "description", "supplementary_file_1",
            "supplementary_file", "source_name_ch1",
        ):
            values = gsm.metadata.get(key, [])

            if isinstance(values, str):
                values = [values]

            for value in values:
                value = str(value).strip()

                if not value:
                    continue

                for alias in (
                    value,
                    Path(value).name,
                    Path(value).stem,
                ):
                    exact_candidates[alias].add(gsm_id)

                for token in _tokens(value):
                    idx[token].add(gsm_id)

    exact = {
        alias: next(iter(sample_ids))
        for alias, sample_ids in exact_candidates.items()
        if len(sample_ids) == 1
    }

    return idx, exact


def reconcile_ids(expr_cols, want_ids, gse):
    # Preserve original column names as mapping keys.
    original = list(expr_cols)
    wanted = [str(value).strip() for value in want_ids]
    want = set(wanted)

    if not want:
        return {}, "FAILED: no requested sample IDs"

    if len(wanted) != len(want):
        raise ValueError(f"{gse}: repeated requested sample IDs")

    if not pd.Index(original).is_unique:
        raise ValueError(f"{gse}: duplicate expression column names")

    cleaned = {column: str(column).strip() for column in original}

    def acceptable(mapping):
        targets = list(mapping.values())

        if len(targets) != len(set(targets)):
            raise ValueError(
                f"{gse}: multiple expression columns map to one GSM"
            )

        return len(targets) >= 0.9 * len(want)

    mapping = {
        column: value
        for column, value in cleaned.items()
        if value in want
    }

    if acceptable(mapping):
        return mapping, "direct_gsm"

    mapping = {}

    for column, value in cleaned.items():
        hits = set(re.findall(r"GSM\d+", value)) & want

        if len(hits) == 1:
            mapping[column] = next(iter(hits))

    if acceptable(mapping):
        return mapping, "embedded_gsm"

    idx, exact = build_gsm_token_index(gse)

    mapping = {
        column: exact[value]
        for column, value in cleaned.items()
        if value in exact and exact[value] in want
    }

    if acceptable(mapping):
        return mapping, "exact_metadata_string"

    mapping = {}
    ambiguous = 0

    for column, value in cleaned.items():
        hits = set()

        for token in _tokens(value):
            if token in idx and len(idx[token]) == 1:
                hits.update(idx[token])

        hits &= want

        if len(hits) == 1:
            mapping[column] = next(iter(hits))
        elif len(hits) > 1:
            ambiguous += 1

    if acceptable(mapping):
        print(
            f"      token match: {len(mapping)}/{len(want)} "
            f"({ambiguous} ambiguous)"
        )
        return mapping, "token_match_VERIFIED"

    if len(original) == len(wanted):
        return {}, (
            "FAILED_ID_MATCH: equal counts do not prove sample order"
        )

    return {}, f"FAILED(cols={len(original)}, want={len(want)})"


def apply_sample_mapping(df, cmap, cohort):
    if not df.columns.is_unique:
        raise ValueError(
            f"{cohort}: duplicate source expression column names"
        )

    if not cmap:
        raise ValueError(f"{cohort}: no sample mapping available")

    target_ids = pd.Index(list(cmap.values()))

    if target_ids.hasnans or not target_ids.is_unique:
        raise ValueError(
            f"{cohort}: missing or repeated target sample IDs"
        )

    missing = [column for column in cmap if column not in df.columns]

    if missing:
        raise ValueError(
            f"{cohort}: mapping references absent columns: {missing[:5]}"
        )

    out = df.loc[:, list(cmap)].rename(columns=cmap)

    if not out.columns.equals(target_ids):
        raise ValueError(f"{cohort}: mapped sample order is incorrect")

    return out


# ---------- Supplementary matrix loader ----------

def load_supp_matrix(gse):
    directory = PATHS["raw_geo"] / gse

    if not directory.exists():
        return None, "no_dir"

    files = [
        path
        for path in directory.iterdir()
        if path.is_file()
        and path.suffix in (".gz", ".txt", ".tsv", ".csv")
        and not path.name.endswith(
            ("soft.gz", "family.soft.gz", ".sdrf.txt", ".idf.txt")
        )
    ]

    for path in sorted(files, key=lambda p: -p.stat().st_size):
        for sep in ["\t", ","]:
            try:
                df = pd.read_csv(
                    path,
                    sep=sep,
                    index_col=0,
                    low_memory=False,
                    compression="gzip" if path.name.endswith(".gz") else None,
                )
                numeric = df.select_dtypes(include=[np.number])

                if numeric.shape[0] > 5000 and numeric.shape[1] >= 10:
                    return numeric, path.name

            except Exception:
                continue

    return None, "no_supp_matrix"


# ---------- GEO sample table loader ----------

def load_series_matrix(gse):
    try:
        g = GEOparse.get_GEO(
            geo=gse,
            destdir=str(PATHS["raw_geo"] / gse),
            silent=True,
        )

        df = g.pivot_samples("VALUE").apply(
            pd.to_numeric, errors="coerce"
        )
        return df, "series_matrix"

    except Exception as exc:
        return None, f"series_matrix_failed: {exc}"


# ---------- GCT loader ----------

def load_gct(gse):
    directory = PATHS["raw_geo"] / gse

    for path in sorted(
        directory.glob("*.gct*"),
        key=lambda p: -p.stat().st_size,
    ):
        try:
            df = pd.read_csv(
                path,
                sep="\t",
                skiprows=2,
                low_memory=False,
            )

            name = "Name" if "Name" in df.columns else df.columns[0]
            out = df.set_index(name).drop(
                columns=["Description"],
                errors="ignore",
            )

            return (
                out.apply(pd.to_numeric, errors="coerce"),
                f"gct:{path.name}",
            )

        except Exception as exc:
            print(f"      GCT parse failed {path.name}: {exc}")

    return None, "no_gct"


# ---------- Per-GSM supplementary file loader ----------

def load_per_gsm(gse, max_workers=8):
    try:
        g = GEOparse.get_GEO(
            geo=gse,
            destdir=str(PATHS["raw_geo"] / gse),
            silent=True,
            include_data=False,
        )
    except Exception as exc:
        return None, f"per_gsm_soft_failed: {exc}"

    urls = {}

    for gsm_id, gsm in g.gsms.items():
        for key in ("supplementary_file_1", "supplementary_file"):
            values = gsm.metadata.get(key, [])

            if isinstance(values, str):
                values = [values]

            for value in values:
                value = str(value).strip()

                if not value or value.upper() == "NONE":
                    continue

                if value.startswith("//"):
                    value = "https:" + value

                value = value.replace("ftp://", "https://")

                if re.search(
                    r"\.(txt|tsv|csv|counts)(\.gz)?$",
                    urlsplit(value).path,
                    re.I,
                ):
                    urls[gsm_id] = value
                    break

    if len(urls) < 5:
        return None, f"per_gsm_no_files (found {len(urls)})"

    outdir = PATHS["raw_geo"] / gse / "per_gsm"
    outdir.mkdir(parents=True, exist_ok=True)

    series = {}
    failed = 0

    for gsm_id, url in tqdm(
        urls.items(),
        desc=f"  {gse} per-GSM",
        leave=False,
    ):
        filename = Path(urlsplit(url).path).name
        dest = outdir / gsm_id / filename
        dest.parent.mkdir(parents=True, exist_ok=True)

        path, status = download_file(
            url,
            dest,
            retries=2,
            timeout=90,
        )

        if path is None:
            failed += 1
            continue

        try:
            compression = "gzip" if path.name.endswith(".gz") else None
            table = None

            for sep in ["\t", ","]:
                candidate = pd.read_csv(
                    path,
                    sep=sep,
                    header=None,
                    comment="#",
                    compression=compression,
                )

                if candidate.shape[1] >= 2:
                    table = candidate
                    break

            if table is None:
                raise ValueError("No two-column expression table found")

            if not pd.api.types.is_numeric_dtype(table.iloc[:, 1]):
                table = pd.read_csv(
                    path,
                    sep=sep,
                    comment="#",
                    compression=compression,
                )

            table = table.iloc[:, :2].copy()
            table.columns = ["gene", gsm_id]

            table = table[
                ~table["gene"].astype(str).str.startswith(
                    ("__", "N_", "no_feature")
                )
            ]

            if table["gene"].isna().any() or table["gene"].duplicated().any():
                raise ValueError("Missing or duplicate gene identifiers")

            series[gsm_id] = pd.to_numeric(
                table.set_index("gene")[gsm_id],
                errors="raise",
            )

        except Exception as exc:
            failed += 1
            print(f"      {gsm_id}: parse failed: {exc}")

    if len(series) < 5:
        return None, f"per_gsm_parse_failed ({failed} failures)"

    if failed:
        print(f"      {gse}: {failed} per-GSM files failed")

    return pd.DataFrame(series), f"per_gsm({len(series)} files)"


# ---------- Orchestrated loader ----------

def load_expression(gse):
    for loader in (
        load_supp_matrix,
        load_gct,
        load_series_matrix,
        load_per_gsm,
    ):
        df, source = loader(gse)

        if (
            df is not None
            and df.shape[0] > 3000
            and df.shape[1] >= 10
        ):
            return df, source

    return None, "all_loaders_failed"


print("Cell 14b loaded: loaders, GPL mapping, and sample-ID helpers ready.")

In [ ]:
# ============================================================
# CELL 15 — EXPRESSION LOADING  (v3)
# ============================================================
try:
    import mygene
except ModuleNotFoundError:
    import mygene

import logging
assert "PATHS" in globals(), "PATHS undefined — run Cell 2 first"
assert "labels_raw" in globals(), "labels_raw undefined — run Cells 7-13 first"
assert "load_supp_matrix" in globals(), "loaders undefined — run Cell 14b first"
assert "_GPL_CACHE" in globals() or True

MG = mygene.MyGeneInfo()
CACHE = PATHS["interim"] / "expr_cache"; CACHE.mkdir(parents=True, exist_ok=True)
logging.getLogger("biothings.client").setLevel(logging.WARNING)
_GPL_CACHE = globals().get("_GPL_CACHE", {})


def _safe_mg(ids, scope):
    try:
        res = MG.querymany(list(ids), scopes=scope, fields="symbol",
                           species="human", as_dataframe=True, df_index=False,
                           verbose=False)
    except Exception as e:
        print(f"      mygene({scope}) failed: {e}")
        return {}
    if not isinstance(res, pd.DataFrame) or "symbol" not in res.columns:
        return {}
    res = res[res.symbol.notna()].drop_duplicates("query")
    return dict(zip(res["query"].astype(str), res["symbol"].astype(str)))


def _gpl_table(gse):
    if gse in _GPL_CACHE:
        return _GPL_CACHE[gse]
    try:
        g = GEOparse.get_GEO(geo=gse, destdir=str(PATHS["raw_geo"]/gse),
                             silent=True, include_data=False)
        tbl = list(g.gpls.values())[0].table
    except Exception as e:
        print(f"      GPL load failed for {gse}: {e}")
        tbl = None
    _GPL_CACHE[gse] = tbl
    return tbl


def _clean_symbol(v):
    if pd.isna(v):
        return None
    s = str(v).strip()
    if s.upper() in ("---", "NA", "NAN", ""):
        return None
    if "///" in s:
        parts = [p.strip() for p in s.split("///") if p.strip()]
        s = parts[0] if parts else ""
    elif "//" in s:
        parts = [p.strip() for p in s.split("//") if p.strip()]
        toks = [p for p in parts if re.fullmatch(r"[A-Za-z0-9\-\.@_]+", p)]
        s = toks[1] if len(toks) > 1 else (toks[0] if toks else "")
    elif "|" in s:
        s = s.split("|")[0].strip()
    s = s.strip()
    return s if s and s.upper() not in ("---", "NA", "NAN") else None





def map_ids_to_symbols(ids, gse):
    ids = [str(i).strip() for i in ids]
    base = [i.split(".")[0] for i in ids]
    strategies = {}

    sym_like = np.mean([bool(re.fullmatch(r"[A-Z][A-Z0-9\-\.@]{1,14}", i))
                        for i in ids[:500]])
    if sym_like > 0.5:
        strategies["already_symbol"] = ({i: i for i in ids}, sym_like)

    if not re.match(r"^ENSG\d+", ids[0] or ""):
        m = _gpl_map(ids, gse)
        if m:
            strategies["gpl"] = (m, len(set(ids) & set(m)) / len(ids))

    if np.mean([bool(re.match(r"^ENSG\d+", i)) for i in ids[:500]]) > 0.3:
        m = _safe_mg(set(base), "ensembl.gene")
        if m:
            strategies["ensembl"] = ({i: m[b] for i, b in zip(ids, base) if b in m},
                                     len(m) / len(set(base)))

    if np.mean([bool(re.fullmatch(r"\d+", i)) for i in ids[:500]]) > 0.5 \
       and strategies.get("gpl", ({}, 0))[1] < 0.5:
        m = _safe_mg(set(ids), "entrezgene")
        if m:
            strategies["entrez"] = (m, len(m) / len(set(ids)))

    if not strategies:
        return [None] * len(ids), "unmapped(0%)"

    how, (mapping, cov) = max(strategies.items(), key=lambda kv: kv[1][1])
    print(f"      symbol strategies: "
          f"{ {k: f'{v[1]:.1%}' for k, v in strategies.items()} }  -> {how}")
    return [mapping.get(i) for i in ids], f"{how}({cov:.0%})"


# ---------------- run, with per-cohort cache ----------------
EXPR, LOAD_LOG = {}, {}
geo_targets = [g for g in labels_raw.cohort.unique()
               if COHORTS[g].get("repo", "GEO") == "GEO"
               and tier_of(g) in ("T1", "T2", "T3")]
print(f"targets ({len(geo_targets)}): {sorted(geo_targets)}\n")

FORCE_RELOAD = []

for gse in geo_targets:
    cf, lf = CACHE / f"{gse}.parquet", CACHE / f"{gse}.json"
    if cf.exists() and lf.exists() and gse not in FORCE_RELOAD:
        EXPR[gse] = pd.read_parquet(cf)
        LOAD_LOG[gse] = json.loads(lf.read_text())
        print(f"[cache] {gse}: {EXPR[gse].shape}")
        continue

    print(f"\n[load] {gse}")
    log = {"gse": gse}
    try:
        df, src = load_supp_matrix(gse)
        if df is None:
            df, src = load_series_matrix(gse)
        if df is None:
            log.update(status="FAILED", source=src)
            print(f"      FAILED: {src}")
            LOAD_LOG[gse] = log
            continue
        log["source"], log["raw_shape"] = src, list(df.shape)

        want = labels_raw.loc[labels_raw.cohort == gse, "sample_id"].tolist()
        cmap, method = reconcile_ids(df.columns, want, gse)
        log["id_method"], log["matched"] = method, len(cmap)
        if not cmap:
            log["status"] = "FAILED_ID_MATCH"
            print("      FAILED: no sample-ID match")
            LOAD_LOG[gse] = log
            continue
        df = apply_sample_mapping(df, cmap, gse)

        syms, how = map_ids_to_symbols(df.index, gse)
        log["symbol_method"] = how
        df.index = syms
        df = df[df.index.notna()]
        if df.shape[0] < 3000:
            log.update(status="FAILED_SYMBOL_MAP", genes=int(df.shape[0]))
            print(f"      FAILED: only {df.shape[0]} genes after mapping")
            LOAD_LOG[gse] = log
            continue
        df = df.groupby(level=0).median()

        pos = df.values[np.isfinite(df.values) & (df.values > 0)]
        med = float(np.median(pos)) if pos.size else 0.0
        log["median_before"] = round(med, 3)
        if med > 50:
            df = np.log2(df.clip(lower=0) + 1); log["log2_applied"] = True
        else:
            log["log2_applied"] = False

        log.update(status="OK", genes=int(df.shape[0]), samples=int(df.shape[1]),
                   median_after=round(float(np.nanmedian(df.values)), 3))
        EXPR[gse] = df
        df.to_parquet(cf)
        lf.write_text(json.dumps(log, indent=2))
        print(f"      OK: {df.shape[0]} genes x {df.shape[1]} samples")

    except NameError as e:
        raise RuntimeError(
            f"{gse}: loader function missing ({e}). Run Cell 14b FIRST. "
            f"This exact error, silently logged, produced the 4-cohort run.")
    except Exception as e:
        import traceback; traceback.print_exc()
        log.update(status=f"EXCEPTION: {type(e).__name__}: {e}")

    LOAD_LOG[gse] = log

# ---------------- summary ----------------
print("\n" + "=" * 100)
print(f"{'cohort':12s} {'status':20s} {'genes':>6s} {'smp':>4s} {'want':>4s} "
      f"{'median':>7s} {'log2':>5s}  {'id_method':22s} symbol_method")
for gse in geo_targets:
    l = LOAD_LOG.get(gse, {})
    want = (labels_raw.cohort == gse).sum()
    print(f"{gse:12s} {str(l.get('status','?'))[:20]:20s} {l.get('genes',0):6d} "
          f"{l.get('samples',0):4d} {want:4d} {l.get('median_after',0):7.2f} "
          f"{str(l.get('log2_applied','')):5s}  {str(l.get('id_method',''))[:22]:22s} "
          f"{l.get('symbol_method','')}")

(PATHS["reports"]/"expression_load_log.json").write_text(json.dumps(LOAD_LOG, indent=2))

ok  = [g for g, l in LOAD_LOG.items() if l.get("status") == "OK"]
bad = [g for g in geo_targets if g not in ok]
nameerr = [g for g, l in LOAD_LOG.items() if "NameError" in str(l.get("status", ""))]

print(f"\nOK {len(ok)}/{len(geo_targets)}: {sorted(ok)}")
print(f"needs repair by Cells 17/19/24: {sorted(bad)}")
assert not nameerr, f"NameError for {nameerr} — Cell 14b did not run"
assert len(ok) >= 5, (
    f"only {len(ok)} cohorts loaded — expected ~6 at this stage. "
    f"Check that Cell 14b ran and the cache was cleared.")
print("\nProceed to Cell 17 (retry) — it repairs the rest.")

In [ ]:
# ============================================================
# CELL 17 — EXPRESSION LOADER v3
# Fixes:
#  (a) empty/absent GPL table -> IndexError
#  (b) SuperSeries: pick the GPL that matches the expression samples
#  (c) per-GSM supplementary counts files (GSE135251, GSE207310)
#  (d) GCT format (GSE193066)
#  (e) token-based ID reconciliation -> removes positional guessing
# ============================================================
from urllib.parse import urlsplit





# ---------- re-run only what failed ----------
RETRY = [g for g, l in LOAD_LOG.items()
         if l.get("status") != "OK" or "positional" in str(l.get("id_method", ""))
         or "8%" in str(l.get("symbol_method", ""))]
print("RETRY:", RETRY)

for gse in RETRY:
    print(f"\n[reload] {gse}")
    log = {"gse": gse}
    try:
        df, src = load_expression(gse)
        if df is None:
            LOAD_LOG[gse] = {**log, "status": "FAILED", "source": src}
            print(f"      FAILED: {src}"); continue
        log["source"], log["raw_shape"] = src, list(df.shape)

        want = labels_raw.loc[labels_raw.cohort == gse, "sample_id"].tolist()
        cmap, method = reconcile_ids(df.columns, want, gse)
        log["id_method"], log["matched"] = method, len(cmap)
        if not cmap:
            LOAD_LOG[gse] = {**log, "status": "FAILED_ID_MATCH"}
            print(f"      FAILED: {method}"); continue
        df = apply_sample_mapping(df, cmap, gse)

        syms, how = map_ids_to_symbols(df.index, gse)
        log["symbol_method"] = how
        df.index = syms
        df = df[df.index.notna()]
        if df.shape[0] < 3000:
            LOAD_LOG[gse] = {**log, "status": "FAILED_SYMBOL_MAP",
                             "genes": int(df.shape[0])}
            print(f"      FAILED: {df.shape[0]} genes after mapping"); continue
        df = df.groupby(level=0).median()

        pos = df.values[np.isfinite(df.values) & (df.values > 0)]
        med = float(np.median(pos)) if pos.size else 0.0
        log["median_before"] = round(med, 3)
        if med > 50:
            df = np.log2(df.clip(lower=0) + 1); log["log2_applied"] = True
        else:
            log["log2_applied"] = False
        log.update(status="OK", genes=int(df.shape[0]), samples=int(df.shape[1]),
                   median_after=round(float(np.nanmedian(df.values)), 3))
        EXPR[gse] = df
        df.to_parquet(CACHE / f"{gse}.parquet")
        (CACHE / f"{gse}.json").write_text(json.dumps(log, indent=2))
        print(f"      OK: {df.shape[0]} genes x {df.shape[1]} samples, "
              f"median={log['median_after']}, ids={method}")
    except Exception as e:
        import traceback; traceback.print_exc()
        log.update(status=f"EXCEPTION: {type(e).__name__}: {e}")
    LOAD_LOG[gse] = log

print("\n" + "="*104)
print(f"{'cohort':12s} {'status':18s} {'genes':>6s} {'smp':>4s} {'want':>4s} "
      f"{'median':>7s}  {'id_method':24s} symbol_method")
for gse in geo_targets:
    l = LOAD_LOG.get(gse, {})
    print(f"{gse:12s} {str(l.get('status','?'))[:18]:18s} {l.get('genes',0):6d} "
          f"{l.get('samples',0):4d} {(labels_raw.cohort==gse).sum():4d} "
          f"{l.get('median_after',0):7.2f}  {str(l.get('id_method',''))[:24]:24s} "
          f"{l.get('symbol_method','')}")

(PATHS["reports"]/"expression_load_log.json").write_text(json.dumps(LOAD_LOG, indent=2))
ok = [g for g, l in LOAD_LOG.items() if l.get("status") == "OK"]
print(f"\nOK {len(ok)}/{len(geo_targets)}   "
      f"still failing: {[g for g in geo_targets if g not in ok]}")
print(f"still positional: "
      f"{[g for g in ok if 'positional' in str(LOAD_LOG[g].get('id_method',''))]}")

In [ ]:
# ============================================================
# CELL 18 — ID VERIFICATION  (v2)
# Cell 16 returned "OK" when there was nothing to check.
# NaN is now UNVERIFIED, not a pass.
# Three independent checks; a cohort must pass at least one.
# ============================================================

def verify_ids(gse):
    checks = {}

    # (1) stage encoded in the sample title
    try:
        g = GEOparse.get_GEO(geo=gse, destdir=str(PATHS["raw_geo"]/gse),
                             silent=True, include_data=False)
        rows = []
        for gsm_id, gsm in g.gsms.items():
            title = " ".join(gsm.metadata.get("title", []) +
                             gsm.metadata.get("description", []))
            m = re.search(r"[_\- ]F([0-4])\b", title)
            n = re.search(r"[_\- ](N|normal)\b", title, re.I)
            rows.append(dict(sample_id=gsm_id,
                             title_stage=f"F{m.group(1)}" if m else
                                         ("healthy" if n else None)))
        t = pd.DataFrame(rows).merge(
            labels_raw[labels_raw.cohort == gse][["sample_id","stage","resolution"]],
            on="sample_id")
        ck = t[t.title_stage.notna()]
        if len(ck) >= 10:
            agree = ((ck.title_stage == ck.stage) |
                     ((ck.title_stage == "healthy") &
                      (ck.resolution == RES_HEALTHY))).mean()
            checks["title_stage"] = (agree, len(ck))
    except Exception as e:
        checks["title_stage_error"] = str(e)

    # (2) biological sanity: COL1A1 must rise with stage
    if gse in EXPR:
        df = EXPR[gse]
        lab = labels_raw[(labels_raw.cohort == gse) & labels_raw.stage.notna()]
        marker = next((m for m in ("COL1A1","COL1A2","COL3A1","TIMP1","ACTA2")
                       if m in df.index), None)
        if marker is not None and len(lab) >= 20:
            s = df.loc[marker].reindex(lab.sample_id.values)
            stg = lab.stage.str[1].astype(float).values
            ok = np.isfinite(s.values) & np.isfinite(stg)
            if ok.sum() >= 20:
                from scipy.stats import spearmanr
                r, p = spearmanr(s.values[ok], stg[ok])
                checks["marker_corr"] = (marker, round(float(r),3), float(p), int(ok.sum()))

    # (3) sex marker vs recorded sex (XIST / RPS4Y1)
    if gse in EXPR:
        df = EXPR[gse]
        d = DISCOVERY.get(gse, {}).get("per_sample", {})
        sexkey = next((k for k in ("sex","gender") if any(k in r for r in d.values())), None)
        gene = next((g for g in ("XIST","RPS4Y1","DDX3Y","KDM5D") if g in df.index), None)
        if sexkey and gene:
            rows = [(s, str(r[sexkey]).lower()[:1]) for s, r in d.items() if sexkey in r]
            sub = pd.DataFrame(rows, columns=["sample_id","sex"])
            sub = sub[sub.sample_id.isin(df.columns)]
            if len(sub) >= 20 and sub.sex.nunique() == 2:
                v = df.loc[gene, sub.sample_id.values].astype(float)
                grp = sub.assign(v=v.values).groupby("sex").v.mean()
                sep = abs(grp.iloc[0] - grp.iloc[1]) / (df.loc[gene].std() + 1e-9)
                checks["sex_marker"] = (gene, round(float(sep),2), dict(grp.round(2)))
    return checks


print(f"{'cohort':12s} {'id_method':24s} verification")
for gse in [g for g, l in LOAD_LOG.items() if l.get("status") == "OK"]:
    method = str(LOAD_LOG[gse].get("id_method",""))
    c = verify_ids(gse)
    bits = []
    verified = method in {
    "direct_gsm",
    "embedded_gsm",
    "exact_metadata_string",
    "token_match_VERIFIED",
    "normalized_match_VERIFIED",
}
    if "title_stage" in c:
        a, n = c["title_stage"]
        bits.append(f"title_stage={a:.0%}(n={n})")

    if "marker_corr" in c:
        m, r, p, n = c["marker_corr"]
        bits.append(f"{m}~stage rho={r} p={p:.1e}")

    if "sex_marker" in c:
        g_, sep, grp = c["sex_marker"]
        bits.append(f"{g_} sex-sep={sep}")

    status = "VERIFIED" if verified else ">>> UNVERIFIED — DO NOT USE IN T1"
    print(f"{gse:12s} {method[:24]:24s} {status}")
    for b in bits:
        print(f"{'':12s} {'':24s}   {b}")

In [ ]:
# CELL 18b — SOURCE PROCESSING REVIEW
# Run this before accepting Cell 19's normalization choices.

review_rows = []

for gse in geo_targets:
    rec = LOAD_LOG.get(gse, {})

    review_rows.append({
        "cohort": gse,
        "status": rec.get("status"),
        "source": rec.get("source"),
        "raw_shape": str(rec.get("raw_shape")),
        "id_method": rec.get("id_method"),
        "median_before": rec.get("median_before"),
        "log2_applied": rec.get("log2_applied"),
        "symbol_method": rec.get("symbol_method"),
    })

print(pd.DataFrame(review_rows).to_string(index=False))

for gse in geo_targets:
    print(f"\n--- {gse}: source processing metadata ---")

    obj = GEOparse.get_GEO(
        geo=gse,
        destdir=str(PATHS["raw_geo"] / gse),
        silent=True,
        include_data=False,
    )

    processing = set()

    for gsm in obj.gsms.values():
        values = gsm.metadata.get("data_processing", [])
        if isinstance(values, str):
            values = [values]

        processing.update(
            str(value).strip()
            for value in values
            if str(value).strip()
        )

    if processing:
        for description in sorted(processing):
            print(description)
    else:
        print("No GSM data_processing metadata found.")

In [ ]:
# ============================================================
# CELL 19 — SCALE DETECTION + TARGETED COHORT FIXES
#   (a) raw counts -> CPM -> log2   (GSE135251, GSE207310)
#   (b) GSE31803: drop methylation samples, diagnose index
#   (c) GSE240729: normalized full-string ID match
#   (d) GSE193066: prefer the 164-sample GCT, match baseline only
# ============================================================

def detect_scale(df):
    """Return one of: raw_counts | log_scale | linear_intensity | tpm_fpkm."""
    v = df.values
    fin = v[np.isfinite(v)]
    if fin.size == 0:
        return "empty", {}
    pos = fin[fin > 0]
    stats = dict(
        max=float(np.nanmax(fin)),
        median_pos=float(np.median(pos)) if pos.size else 0.0,
        p99=float(np.percentile(fin, 99)),
        frac_zero=float((fin == 0).mean()),
        frac_integer=float(np.mean(np.isclose(pos[:200000] % 1, 0))) if pos.size else 0.0,
    )
    if stats["frac_integer"] > 0.95 and stats["max"] > 1000:
        return "raw_counts", stats
    if stats["max"] > 100 and stats["frac_integer"] < 0.95:
        return "tpm_fpkm", stats
    if stats["max"] <= 30:
        return "log_scale", stats
    return "linear_intensity", stats


def normalize_scale(df, gse):
    kind, s = detect_scale(df)
    print(f"      scale={kind}  max={s.get('max',0):.1f} "
          f"median_pos={s.get('median_pos',0):.2f} "
          f"zeros={s.get('frac_zero',0):.1%} int={s.get('frac_integer',0):.0%}")
    if kind == "raw_counts":
        lib = df.sum(axis=0).replace(0, np.nan)
        df = np.log2(df.div(lib, axis=1) * 1e6 + 1)          # CPM -> log2
        note = "CPM+log2"
    elif kind == "tpm_fpkm":
        df = np.log2(df.clip(lower=0) + 1)
        note = "log2"
    elif kind == "linear_intensity":
        df = np.log2(df.clip(lower=0) + 1)
        note = "log2"
    else:
        note = "none(already log)"
    # low-expression filter: keep genes detected in >=20% of samples
    keep = (df > 0).mean(axis=1) >= 0.20
    print(f"      {note}; gene filter {int(keep.sum())}/{len(keep)} kept; "
          f"new median={float(np.nanmedian(df[keep].values)):.2f}")
    return df[keep], f"{kind}->{note}"


# ---------- (c) normalized-string ID matching ----------
def _norm_key(s):
    return re.sub(r"[^a-z0-9]", "", str(s).lower())

def reconcile_normalized(expr_cols, want_ids, gse):
    idx, exact = build_gsm_token_index(gse)
    norm2gsm = defaultdict(set)
    for raw, gsm in exact.items():
        k = _norm_key(raw)
        if len(k) >= 6:
            norm2gsm[k].add(gsm)
            norm2gsm[re.sub(r"\.(txt|tsv|csv|gz|counts)+$", "", k)].add(gsm)
    want = set(map(str, want_ids))
    out = {}
    for c in expr_cols:
        k = _norm_key(c)
        cands = set()
        if k in norm2gsm:
            cands = set(norm2gsm[k])
        else:                                  # substring containment
            for nk, gs in norm2gsm.items():
                if len(k) >= 8 and (k in nk or nk in k):
                    cands |= gs
        cands &= want
        if len(cands) == 1:
            out[str(c)] = cands.pop()
    return out


# =========================== APPLY ===========================

# ---- (a) rescale everything that needs it ----
print("SCALE NORMALISATION")
for gse in list(EXPR.keys()):
    print(f"  {gse}")
    EXPR[gse], note = normalize_scale(EXPR[gse], gse)
    LOAD_LOG[gse]["scale_fix"] = note
    LOAD_LOG[gse]["genes"] = int(EXPR[gse].shape[0])
    LOAD_LOG[gse]["median_after"] = round(float(np.nanmedian(EXPR[gse].values)), 3)
    EXPR[gse].to_parquet(CACHE / f"{gse}.parquet")

# ---- (b) GSE31803: diagnose index, drop methylation ----
print("\nGSE31803 DIAGNOSIS")
if "GSE31803" in EXPR:
    d = EXPR["GSE31803"]
    print("  index sample:", list(d.index[:8]))
    keep = labels_raw[(labels_raw.cohort == "GSE31803") &
                      (labels_raw.resolution != RES_UNKNOWN)].sample_id.tolist()
    have = [c for c in d.columns if c in keep]
    print(f"  expression samples with usable stage: {len(have)}/{d.shape[1]}")
    if 60 <= len(have) <= 80:
        EXPR["GSE31803"] = d[have]
        LOAD_LOG["GSE31803"]["samples"] = len(have)
        LOAD_LOG["GSE31803"]["note"] = "methylation samples dropped"
        EXPR["GSE31803"].to_parquet(CACHE / "GSE31803.parquet")
        print(f"  -> kept {len(have)} expression samples")
    else:
        print("  !! unexpected count — inspect before proceeding")

# ---- (c) GSE240729 ----
print("\nGSE240729 ID RE-MATCH")
if "GSE240729" in EXPR:
    raw, src = load_expression("GSE240729")
    if raw is not None:
        want = labels_raw.loc[labels_raw.cohort == "GSE240729", "sample_id"].tolist()
        cmap = reconcile_normalized(raw.columns, want, "GSE240729")
        print(f"  normalized match: {len(cmap)}/{len(want)}")
        print(f"  sample cols: {list(raw.columns[:4])}")
        if len(cmap) >= 0.9 * len(want):
            df = apply_sample_mapping(raw, cmap, "GSE240729")
            syms, how = map_ids_to_symbols(df.index, "GSE240729")
            df.index = syms; df = df[df.index.notna()].groupby(level=0).median()
            df, note = normalize_scale(df, "GSE240729")
            EXPR["GSE240729"] = df
            LOAD_LOG["GSE240729"].update(
                id_method="normalized_match_VERIFIED", matched=len(cmap),
                genes=int(df.shape[0]), samples=int(df.shape[1]), scale_fix=note,
                median_after=round(float(np.nanmedian(df.values)), 3))
            df.to_parquet(CACHE / "GSE240729.parquet")
            print("  -> RESOLVED, no longer positional")
        else:
            print("  -> still unmatched; stays positional_UNVERIFIED")

# ---- (d) GSE193066 ----
print("\nGSE193066 RETRY")
gdir = PATHS["raw_geo"] / "GSE193066"
gcts = sorted(gdir.glob("*.gct*"), key=lambda f: -f.stat().st_size)
print("  GCT files:", [f.name for f in gcts])
baseline = labels_raw[(labels_raw.cohort == "GSE193066") &
                      (labels_raw.resolution != "duplicate_biopsy")].sample_id.tolist()
print(f"  baseline samples wanted: {len(baseline)}")

for f in gcts:
    try:
        df = pd.read_csv(f, sep="\t", skiprows=2, low_memory=False)
        name = "Name" if "Name" in df.columns else df.columns[0]
        df = df.set_index(name).drop(columns=["Description"], errors="ignore")
        df = df.apply(pd.to_numeric, errors="coerce").dropna(how="all")
        print(f"  {f.name}: {df.shape}, cols {list(df.columns[:3])}")
        cmap, method = reconcile_ids(df.columns, baseline, "GSE193066")
        if len(cmap) < 0.9 * len(baseline):
            cmap = reconcile_normalized(df.columns, baseline, "GSE193066")
            method = "normalized_match_VERIFIED"
        print(f"    matched {len(cmap)}/{len(baseline)} via {method}")
        if len(cmap) >= 0.9 * len(baseline):
            d = apply_sample_mapping(df, cmap, "GSE193066")
            syms, how = map_ids_to_symbols(d.index, "GSE193066")
            d.index = syms; d = d[d.index.notna()].groupby(level=0).median()
            d, note = normalize_scale(d, "GSE193066")
            EXPR["GSE193066"] = d
            LOAD_LOG["GSE193066"] = dict(
                gse="GSE193066", status="OK", source=f"gct:{f.name}",
                id_method=method, matched=len(cmap), symbol_method=how,
                scale_fix=note, genes=int(d.shape[0]), samples=int(d.shape[1]),
                median_after=round(float(np.nanmedian(d.values)), 3))
            d.to_parquet(CACHE / "GSE193066.parquet")
            print("    -> RESOLVED")
            break
    except Exception as e:
        print(f"  {f.name}: {type(e).__name__}: {e}")

# ---------------- final summary ----------------
print("\n" + "="*112)
print(f"{'cohort':12s} {'status':10s} {'genes':>6s} {'smp':>4s} {'want':>4s} "
      f"{'median':>7s}  {'scale_fix':22s} {'id_method':26s}")
for gse in geo_targets:
    l = LOAD_LOG.get(gse, {})
    want = labels_raw[(labels_raw.cohort == gse) &
                      (~labels_raw.resolution.isin([RES_UNKNOWN, "duplicate_biopsy"]))].shape[0]
    print(f"{gse:12s} {str(l.get('status','?'))[:10]:10s} {l.get('genes',0):6d} "
          f"{l.get('samples',0):4d} {want:4d} {l.get('median_after',0):7.2f}  "
          f"{str(l.get('scale_fix',''))[:22]:22s} {str(l.get('id_method',''))[:26]:26s}")

(PATHS["reports"]/"expression_load_log.json").write_text(json.dumps(LOAD_LOG, indent=2))
ok = [g for g, l in LOAD_LOG.items() if l.get("status") == "OK"]
print(f"\nOK {len(ok)}/{len(geo_targets)}")
print(f"still positional: "
      f"{[g for g in ok if 'positional' in str(LOAD_LOG[g].get('id_method',''))]}")
meds = [l["median_after"] for l in LOAD_LOG.values()
        if l.get("status") == "OK" and l.get("median_after")]
print(f"median range across cohorts: {min(meds):.2f} to {max(meds):.2f}")
print("  ^ this spread is the platform-scale problem. It is EXPECTED and is exactly")
print("    why the GTEx deviation feature must be rank-based, not z-scored.")

In [ ]:
# ============================================================
# CELL 20 — SAMPLE-LEVEL QC
#  (a) library-size / depth normalisation check per cohort
#  (b) outlier samples by median correlation to cohort centroid
#  (c) marker sanity (COL1A1 / ALB) per cohort
#  Nothing is dropped automatically — this reports, you decide.
# ============================================================
from scipy.stats import spearmanr

def qc_cohort(gse, df):
    v = df.values
    col_med = np.nanmedian(v, axis=0)
    col_sum = np.nansum(v, axis=0)
    q = dict(
        n_genes=int(df.shape[0]), n_samples=int(df.shape[1]),
        median_of_medians=round(float(np.nanmedian(col_med)), 3),
        median_iqr=round(float(np.nanpercentile(col_med, 75) -
                               np.nanpercentile(col_med, 25)), 3),
        median_cv=round(float(np.nanstd(col_med) / (abs(np.nanmean(col_med)) + 1e-9)), 3),
        sum_cv=round(float(np.nanstd(col_sum) / (abs(np.nanmean(col_sum)) + 1e-9)), 3),
        frac_zero=round(float(np.mean(v == 0)), 3),
    )
    # depth-normalised data has near-constant per-sample totals
    q["depth_normalised"] = bool(q["sum_cv"] < 0.10)

    # correlation of each sample to the cohort centroid
    sub = df.sample(min(3000, df.shape[0]), random_state=42)
    c = np.corrcoef(np.nan_to_num(sub.values.T))
    med_corr = np.median(c, axis=1)
    q["min_sample_corr"] = round(float(med_corr.min()), 3)
    q["n_outliers_r<0.7"] = int((med_corr < 0.7).sum())
    q["outlier_ids"] = list(df.columns[med_corr < 0.7][:8])
    for g in ("COL1A1", "ALB"):
        q[f"{g}_present"] = bool(g in df.index)
    return q


QC = {}
print(f"{'cohort':12s} {'genes':>6s} {'smp':>4s} {'medOfMed':>8s} {'medIQR':>7s} "
      f"{'sumCV':>6s} {'depthN':>6s} {'zero%':>6s} {'minR':>6s} {'out':>4s}")
for gse, df in EXPR.items():
    q = qc_cohort(gse, df); QC[gse] = q
    print(f"{gse:12s} {q['n_genes']:6d} {q['n_samples']:4d} "
          f"{q['median_of_medians']:8.2f} {q['median_iqr']:7.2f} "
          f"{q['sum_cv']:6.3f} {str(q['depth_normalised']):>6s} "
          f"{q['frac_zero']*100:5.1f}% {q['min_sample_corr']:6.3f} "
          f"{q['n_outliers_r<0.7']:4d}")

print("\nCOHORTS NEEDING DEPTH NORMALISATION (sum_cv >= 0.10):")
need = [g for g, q in QC.items() if not q["depth_normalised"]]
for g in need:
    print(f"  {g}: sum_cv={QC[g]['sum_cv']:.3f}  median_iqr={QC[g]['median_iqr']:.2f}")
print("  ->", need or "none")

# ---- quantile-normalise within cohort where depth varies ----
# Rank-preserving, within-cohort only, no labels, no cross-cohort info.
def quantile_normalize(df):
    ranks = df.rank(axis=0, method="average")
    ref = np.nanmean(np.sort(df.values, axis=0), axis=1)
    out = np.empty_like(df.values, dtype=float)
    n = df.shape[0]
    for j in range(df.shape[1]):
        idx = np.clip((ranks.values[:, j] - 1).astype(int), 0, n - 1)
        out[:, j] = ref[idx]
    return pd.DataFrame(out, index=df.index, columns=df.columns)

APPLY_QN = True
if APPLY_QN and need:
    for g in need:
        EXPR[g] = quantile_normalize(EXPR[g])
        LOAD_LOG[g]["within_cohort_qn"] = True
        EXPR[g].to_parquet(CACHE / f"{g}.parquet")
        q = qc_cohort(g, EXPR[g])
        print(f"  {g}: sum_cv {QC[g]['sum_cv']:.3f} -> {q['sum_cv']:.3f}")
        QC[g] = q

print("\nOUTLIER SAMPLES (median correlation to centroid < 0.7):")
for g, q in QC.items():
    if q["n_outliers_r<0.7"]:
        print(f"  {g}: {q['n_outliers_r<0.7']} -> {q['outlier_ids']}")
print("  Phase 2 performed no outlier removal. Keeping them is defensible;")
print("  report the count either way. Do NOT drop silently.")

(PATHS["reports"]/"qc_samples.json").write_text(json.dumps(QC, indent=2, default=str))

In [ ]:
# ============================================================
# CELL 22 — IDENTIFIER DIAGNOSIS + REMAP
# BUG: the "already_symbol" regex matched ENSG00000123456
#      (15 chars fits [A-Z][A-Z0-9\-\.@]{1,14}), so Ensembl-indexed
#      cohorts skipped mapping and shared 0 genes with symbol cohorts.
# ============================================================

def classify_index(idx, n=500):
    s = [str(i) for i in list(idx)[:n]]
    f = lambda p: float(np.mean([bool(re.match(p, x)) for x in s]))
    return dict(
        ensembl   = f(r"^ENS[GTP]\d"),
        refseq    = f(r"^(NM_|NR_|XM_|XR_|NP_)\d"),
        numeric   = f(r"^\d+$"),
        symbol    = float(np.mean([_looks_like_symbol(x) for x in s])),
        has_dot   = f(r"^[^.]+\.\d+$"),
    )

def _looks_like_symbol(i):
    i = str(i)
    if re.match(r"^ENS[GTP]\d", i):                 return False   # <-- the fix
    if re.match(r"^(NM_|NR_|XM_|XR_|NP_)\d", i):    return False
    if re.fullmatch(r"\d+(\.\d+)?", i):             return False
    if len(i) > 15:                                 return False
    return bool(re.fullmatch(r"[A-Za-z][A-Za-z0-9\-\.@_]{0,14}", i))


print("INDEX TYPE PER COHORT")
print(f"{'cohort':12s} {'n':>6s} {'ensembl':>8s} {'refseq':>7s} {'numeric':>8s} "
      f"{'symbol':>7s}  first 4 index values")
IDX_TYPE = {}
for gse, df in EXPR.items():
    c = classify_index(df.index)
    kind = max(("ensembl","refseq","numeric","symbol"), key=lambda k: c[k])
    IDX_TYPE[gse] = kind if c[kind] > 0.5 else "mixed"
    print(f"{gse:12s} {df.shape[0]:6d} {c['ensembl']:8.1%} {c['refseq']:7.1%} "
          f"{c['numeric']:8.1%} {c['symbol']:7.1%}  {list(df.index[:4])}")
print("\nclassified:", IDX_TYPE)

# ---------- pairwise overlap BEFORE the fix ----------
def overlap_matrix(expr, title):
    ks = list(expr.keys())
    M = pd.DataFrame(0, index=ks, columns=ks)
    for a in ks:
        for b in ks:
            M.loc[a, b] = len(set(expr[a].index) & set(expr[b].index))
    print(f"\nPAIRWISE GENE OVERLAP — {title}")
    print(M.to_string())
    return M
overlap_matrix(EXPR, "BEFORE remap")

# ---------- remap anything that is not symbols ----------
NEEDS_REMAP = [g for g, k in IDX_TYPE.items() if k != "symbol"]
print(f"\nREMAPPING: {NEEDS_REMAP}")

def force_symbol_map(df, gse, kind):
    ids  = [str(i) for i in df.index]
    base = [i.split(".")[0] for i in ids]
    if kind == "ensembl":
        m = _safe_mg(sorted(set(base)), "ensembl.gene")
        mapped = [m.get(b) for b in base]
    elif kind == "refseq":
        m = _safe_mg(sorted(set(base)), "refseq")
        mapped = [m.get(b) for b in base]
    elif kind == "numeric":
        m = _gpl_map(ids, gse)
        if len(set(ids) & set(m)) / len(ids) < 0.3:
            m = _safe_mg(sorted(set(ids)), "entrezgene")
        mapped = [m.get(i) for i in ids]
    else:                                    # mixed: try each, keep best
        best, cov = None, 0
        for sc, key in (("ensembl.gene", base), ("entrezgene", ids), ("symbol", ids)):
            m = _safe_mg(sorted(set(key)), sc)
            c = sum(1 for k in key if k in m) / len(key)
            if c > cov:
                best, cov = (m, key), c
        mapped = [best[0].get(k) for k in best[1]] if best else [None]*len(ids)
    out = df.copy(); out.index = mapped
    out = out[out.index.notna()]
    hit = out.shape[0] / max(df.shape[0], 1)
    return out.groupby(level=0).median(), hit

for gse in NEEDS_REMAP:
    before = EXPR[gse].shape
    fixed, hit = force_symbol_map(EXPR[gse], gse, IDX_TYPE[gse])
    print(f"  {gse:12s} {IDX_TYPE[gse]:8s} {before[0]:6d} -> {fixed.shape[0]:6d} "
          f"symbols ({hit:.0%} mapped)")
    if fixed.shape[0] >= 8000:
        EXPR[gse] = fixed
        LOAD_LOG[gse]["symbol_method"] = f"remapped_from_{IDX_TYPE[gse]}({hit:.0%})"
        LOAD_LOG[gse]["genes"] = int(fixed.shape[0])
        fixed.to_parquet(CACHE / f"{gse}.parquet")
    else:
        print(f"    !! only {fixed.shape[0]} symbols — NOT applied, investigate")

M = overlap_matrix(EXPR, "AFTER remap")
print("\nsanity: every off-diagonal cell should now be >8000")
low = [(a, b, int(M.loc[a, b])) for a in M.index for b in M.columns
       if a != b and M.loc[a, b] < 8000]
print("LOW-OVERLAP PAIRS:", low or "none")

In [ ]:
# ============================================================
# CELL 24 — GSE48452 SYMBOL REPAIR
# GPL11532 gene_assignment format:
#   "acc // SYMBOL // description // cytoband // entrez /// acc // SYMBOL2 // ..."
# Two-level split: '///' between assignments, '//' between fields.
# The symbol is field index 1.
# ============================================================

def parse_gene_assignment(v):
    """Return the first valid HGNC-looking symbol, or None."""
    if pd.isna(v):
        return None
    s = str(v).strip()
    if s in ("---", "", "NA", "nan"):
        return None
    for block in s.split("///"):                     # each assignment
        fields = [f.strip() for f in block.split("//")]
        if len(fields) >= 2:
            sym = fields[1]
            if re.fullmatch(r"[A-Za-z][A-Za-z0-9\-\.@_]{0,14}", sym) \
               and not re.fullmatch(r"\d+[pq][\d\.]+", sym):   # reject cytobands
                return sym
        # single-token fallback
        if len(fields) == 1 and re.fullmatch(r"[A-Z][A-Z0-9\-]{1,14}", fields[0]):
            return fields[0]
    return None


print("GSE48452 REPAIR")
tables = _gpl_tables("GSE48452")
print("  GPL tables:", [(g, t.shape, list(t.columns)[:6]) for g, t in tables])

repaired = None
for gpl_id, tbl in tables:
    if "gene_assignment" not in [c.lower() for c in tbl.columns]:
        continue
    col = next(c for c in tbl.columns if c.lower() == "gene_assignment")
    id_col = "ID" if "ID" in tbl.columns else tbl.columns[0]
    print(f"  using {gpl_id}:{col}, {len(tbl)} probes")
    for ex in tbl[col].dropna().head(3):
        print(f"    raw: {str(ex)[:110]}")
        print(f"    ->   {parse_gene_assignment(ex)}")

    probe2sym = {str(k): parse_gene_assignment(v)
                 for k, v in zip(tbl[id_col], tbl[col])}
    probe2sym = {k: v for k, v in probe2sym.items() if v}
    print(f"  {len(probe2sym)} probes mapped to symbols "
          f"({len(set(probe2sym.values()))} unique)")

    # reload raw so we start from probe IDs, not the corrupted index
    raw, src = load_expression("GSE48452")
    want = labels_raw.loc[labels_raw.cohort == "GSE48452", "sample_id"].tolist()
    cmap, method = reconcile_ids(raw.columns, want, "GSE48452")
    d = apply_sample_mapping(raw, cmap, "GSE48452")
    print(f"  raw index sample: {list(d.index[:4])}")
    cov = len(set(map(str, d.index)) & set(probe2sym)) / len(d.index)
    print(f"  probe coverage: {cov:.1%}")
    if cov < 0.3:
        print("  !! probe IDs do not match the GPL table — inspect above")
        break

    d.index = [probe2sym.get(str(i)) for i in d.index]
    d = d[d.index.notna()].groupby(level=0).median()
    d, note = normalize_scale(d, "GSE48452")
    repaired = d
    print(f"  -> {d.shape[0]} symbols x {d.shape[1]} samples")
    break

if repaired is not None and repaired.shape[0] >= 8000:
    EXPR["GSE48452"] = repaired
    LOAD_LOG["GSE48452"].update(symbol_method="gene_assignment_parsed",
                                genes=int(repaired.shape[0]),
                                samples=int(repaired.shape[1]))
    repaired.to_parquet(CACHE / "GSE48452.parquet")
    others = [g for g in EXPR if g != "GSE48452"]
    ov = {g: len(set(repaired.index) & set(EXPR[g].index)) for g in others}
    print(f"\n  overlap with other cohorts: min={min(ov.values())}, "
          f"median={int(np.median(list(ov.values())))}")
    print("  ", {k: v for k, v in sorted(ov.items(), key=lambda x: x[1])[:4]})
else:
    print("\n  REPAIR FAILED — GSE48452 is T2 (training-only) so the pipeline can")
    print("  proceed without it. Set DROP_GSE48452=True in the next cell if so.")

In [ ]:
# ============================================================
# CELL 24b — RECOVER GSE240729
# Run after Cell 24 and before Cell 25.
# ============================================================

gse = "GSE240729"
raw, source = load_expression(gse)

if raw is None:
    raise RuntimeError(f"{gse}: expression matrix could not be loaded")

want = labels_raw.loc[
    labels_raw.cohort.eq(gse), "sample_id"
].astype(str).tolist()

cmap = reconcile_normalized(raw.columns, want, gse)

if (
    len(want) != 67
    or len(cmap) != len(want)
    or set(cmap.values()) != set(want)
):
    raise ValueError(
        f"{gse}: expected a complete, unique mapping of 67 samples; "
        f"got {len(cmap)} matches"
    )

counts = apply_sample_mapping(raw, cmap, gse)
values = counts.to_numpy(dtype=float)

if not np.isfinite(values).all():
    raise ValueError(f"{gse}: raw counts contain NaN or infinity")

if (values < 0).any():
    raise ValueError(f"{gse}: raw counts contain negative values")

integer_fraction = float(
    np.isclose(values, np.rint(values), atol=1e-8).mean()
)

if integer_fraction < 0.99:
    raise ValueError(
        f"{gse}: source does not look like raw integer counts "
        f"({integer_fraction:.2%} integer-like); inspect its units"
    )

library_size = counts.sum(axis=0)

if (library_size <= 0).any():
    raise ValueError(f"{gse}: a sample has zero total counts")

# Normalize raw counts before aggregating gene identifiers.
log_cpm = np.log2(
    counts.div(library_size, axis=1) * 1_000_000 + 1
)

index_stats = classify_index(log_cpm.index)
kind = max(
    ("ensembl", "refseq", "numeric", "symbol"),
    key=lambda name: index_stats[name],
)

if index_stats[kind] <= 0.5:
    kind = "mixed"

if kind == "symbol":
    mapped = log_cpm.copy()
    mapped.index = mapped.index.astype(str).str.strip()
    mapped = mapped.groupby(level=0).median()
    mapping_method = "existing_gene_symbols"
else:
    mapped, hit = force_symbol_map(log_cpm, gse, kind)
    mapping_method = f"remapped_from_{kind}({hit:.1%})"

mapped = mapped.loc[
    (mapped > 0).mean(axis=1) >= 0.20
].copy()

check_expression(mapped, f"{gse}: normalized expression")

if mapped.shape[0] < 8000:
    raise ValueError(
        f"{gse}: only {mapped.shape[0]} mapped genes; inspect mapping"
    )

if set(mapped.columns) != set(want):
    raise ValueError(f"{gse}: normalized sample IDs do not match labels")

EXPR[gse] = mapped
IDX_TYPE[gse] = "symbol"

LOAD_LOG[gse] = {
    "gse": gse,
    "status": "OK",
    "source": source,
    "raw_shape": list(raw.shape),
    "id_method": "normalized_match_VERIFIED",
    "matched": len(cmap),
    "symbol_method": mapping_method,
    "scale_fix": "raw_counts->CPM+log2",
    "genes": int(mapped.shape[0]),
    "samples": int(mapped.shape[1]),
}

mapped.to_parquet(CACHE / f"{gse}.parquet")
(CACHE / f"{gse}.json").write_text(
    json.dumps(LOAD_LOG[gse], indent=2)
)

print(
    f"{gse} recovered: {mapped.shape[0]} genes x "
    f"{mapped.shape[1]} samples; {mapping_method}"
)

In [ ]:
# Pin the per-cohort inputs: hash every cached matrix so a later run can
# prove it starts from the same expression data.
_man = {p.name: sha256_file(p) for p in sorted((PATHS['interim'] / 'expr_cache').glob('*.parquet'))}
(PATHS['reports'] / 'expr_cache_manifest.json').write_text(json.dumps(_man, indent=2))
print(f'{len(_man)} cohort matrices hashed')

In [ ]:
# Optional: compare the rebuilt labels with the archived ones before going on.
ARCHIVE_LABELS = ROOT / 'archive_reference' / 'labels.parquet'   # shared/labels.parquet from Archive.zip
if ARCHIVE_LABELS.exists():
    _a = pd.read_parquet(ARCHIVE_LABELS)
    _b = labels_raw.set_index('sample_id').reindex(_a.index)
    _cols = ['cohort', 'stage', 'resolution', 'y_T1_significant', 'y_T2_ordinal', 'y_T3_early_adv']
    _diff = {c: int((_a[c].astype(str) != _b[c].astype(str)).sum()) for c in _cols}
    print('records missing from this run:', int(_b.cohort.isna().sum()), ' label differences:', _diff)
else:
    print('archive labels not found; skipped')

## Part 2: integration, references, graph, features, folds, export

In [ ]:
import sys; sys.path.insert(0, str(ROOT / 'code'))   # folder holding coda_preproc/
from coda_preproc import config as C, integrate, reference, graph, features, folds, export

platform_of = {g: COHORTS[g]['platform'] for g in EXPR}
bulk_all, meta_all = integrate.integrate_all(EXPR, platform_of, labels_raw)
bulk_sup, sup, healthy = integrate.split_supervised(bulk_all, meta_all)
bulk_raw = integrate.uncorrected_like(EXPR, bulk_sup)
print('bulk', bulk_all.shape, 'supervised', bulk_sup.shape, 'controls', healthy.shape)

In [ ]:
gtex = reference.load_gtex_liver(PATHS['raw_gtex'] / 'gtex_v8_gene_tpm.gct.gz',
                                 PATHS['raw_gtex'] / 'gtex_v8_sample_attributes.txt')
disp = reference.rank_displacement(bulk_all, gtex)
dev_pg = reference.z_in_sample(disp)
dev_targets, dev_info = reference.deviation_variants(disp, meta_all)
print('GTEx', gtex.shape, dev_info)

In [ ]:
panel, dropped = graph.select_panel(bulk_sup)
print(f'panel {len(panel)}; excluded by alphabetical truncation: {dropped}')
edges, topo = graph.string_subgraph(PATHS['raw_string'] / 'string_links.txt.gz',
                                    PATHS['raw_string'] / 'string_info.txt.gz', panel)
print('directed edges', len(edges))

In [ ]:
order = [s for s in sup.index]
T, order, z = features.build_tensor_AB(bulk_sup[order], sup.loc[order], panel, topo, gtex, dev_pg)
F = folds.make_folds(sup.loc[order], C.T1_COHORTS)
tensors = {f['fold']: folds.fold_tensors(f, T, order, z, sup.loc[order], panel) for f in F}
export.write_package(PATHS['root'] / 'handoff', bulk_sup=bulk_sup[order], bulk_raw=bulk_raw[order],
                     healthy=healthy, sup=sup.loc[order], dev_targets=dev_targets,
                     panel_tab=graph.panel_table(panel), edges=edges, topo=topo,
                     order=order, folds=F, tensors=tensors)
print('handoff written')